# Network Intrusion Detection — Progressive Dataset Evaluation

**Paper:** Chua & Salam (2023), *Evaluation of ML Algorithms in Network-Based Intrusion Detection Using Progressive Dataset*, Symmetry 15, 1251

**Setup:** Run this header cell first every time you open a new Colab session.

In [ ]:
# ── Header cell: run this first in every new Colab session ──────────────────
import sys, os

# 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# 2. Clone repo so src/ modules are importable
REPO_URL = 'https://github.com/Rosette28/data-science-cyber-final-project'  # ← update
REPO_DIR = '/content/ids-project'
if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
# Also add src/ so bare imports (import models, import preprocessing) work
if os.path.join(REPO_DIR, 'src') not in sys.path:
    sys.path.insert(0, os.path.join(REPO_DIR, 'src'))

# 3. Install dependencies
!pip install -q -r {REPO_DIR}/requirements.txt

print('Environment ready.')

In [ ]:
# ── Global configuration — only line you change between runs ─────────────────
DATA_DIR = '/content/drive/MyDrive/ids_data/raw/'  # ← set to your Drive folder

SEED = 42
SUBSAMPLE_FRAC = 0.10  # 10% of each day's CSV, read at load time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from joblib import dump, load

np.random.seed(SEED)
pd.set_option('display.max_columns', 50)
sns.set_theme(style='whitegrid', palette='tab10')

print(f'DATA_DIR = {DATA_DIR}')

---
## §1 — Data Loading & Initial Inspection

Load CIC-IDS2017 (train) and CSE-CIC-IDS2018 (progressive test), keeping ~10% via chunked reading. Inspect shape, dtypes, memory usage, column names, and temporal structure.

In [ ]:
from src.data_loading import load_cic2017, load_cic2018, align_schemas

df_train = load_cic2017(DATA_DIR, subsample_frac=SUBSAMPLE_FRAC, seed=SEED)
df_test  = load_cic2018(DATA_DIR, subsample_frac=SUBSAMPLE_FRAC, seed=SEED)
df_train, df_test = align_schemas(df_train, df_test)

print('Train shape:', df_train.shape)
print('Test  shape:', df_test.shape)

### Shape, dtypes, memory, and column name inspection

In [ ]:
# ── Shape and memory ──────────────────────────────────────────────────────────
for name, df in [('Train (CIC-IDS2017)', df_train), ('Test  (CSE-CIC-IDS2018)', df_test)]:
    mem_mb = df.memory_usage(deep=True).sum() / 1e6
    print(f"{name}: {df.shape[0]:>7,} rows × {df.shape[1]} cols  |  {mem_mb:.1f} MB")

print()

# ── Dtype breakdown ───────────────────────────────────────────────────────────
print("Train dtype counts:")
print(df_train.dtypes.value_counts().to_string())
print("\nTest dtype counts:")
print(df_test.dtypes.value_counts().to_string())

In [ ]:
# ── Column name analysis ──────────────────────────────────────────────────────
# Features fall into five semantic groups derived from CICFlowMeter's documentation.
feature_groups = {
    'Packet length stats':    [c for c in df_train.columns if 'Packet Length' in c or 'Pkt Len' in c or 'Packet Size' in c or 'Segment Size' in c],
    'Packet counts / rates':  [c for c in df_train.columns if 'Packets' in c or 'Pkts' in c or 'Bytes' in c or 'Bulk' in c or 'Subflow' in c],
    'Inter-arrival times':    [c for c in df_train.columns if 'IAT' in c or 'Flow Duration' in c],
    'TCP flags':              [c for c in df_train.columns if 'Flag' in c or 'Win' in c],
    'Other / port / misc':    [c for c in df_train.columns if c not in sum([
        [c for c in df_train.columns if 'Packet Length' in c or 'Pkt Len' in c or 'Packet Size' in c or 'Segment Size' in c],
        [c for c in df_train.columns if 'Packets' in c or 'Pkts' in c or 'Bytes' in c or 'Bulk' in c or 'Subflow' in c],
        [c for c in df_train.columns if 'IAT' in c or 'Flow Duration' in c],
        [c for c in df_train.columns if 'Flag' in c or 'Win' in c],
        ['Label']
    ], []) and c != 'Label'],
}

print("Feature groups (76 features total after schema alignment):\n")
for group, cols in feature_groups.items():
    print(f"  {group} ({len(cols)}): {cols}")

print(f"\nLabel column: 'Label' — unique values in train: {df_train['Label'].unique()}")
print(f"                       — unique values in test:  {df_test['Label'].unique()}")

**Shape and memory** (after load + schema alignment, before deduplication):
- Train (CIC-IDS2017, 10% sample): ~281k rows × 77 cols, ~160 MB
- Test (CSE-CIC-IDS2018, 10% sample): ~125k rows × 77 cols, ~71 MB
- After hygiene cleaning (§1.3): **266,739 × 67 train / 106,906 × 67 test, 66 features**
- Both sets fit comfortably in free Colab RAM (~12 GB limit).

**Dtypes:** 52 `int64` (flag counts, raw packet/byte totals) · 24 `float32` (rates, means, stds — downcast from float64 to halve memory) · 1 `object` (Label).

**66 features across 5 semantic groups:**
- **Packet length stats:** min/max/mean/std of forward and backward packet sizes. Captures *what is being sent* — flooding attacks use fixed-size packets; exfiltration sends large payloads.
- **Packet counts / byte rates:** total packet counts, flow rates (bytes/s, packets/s), subflow counts. Captures *volume and asymmetry* — DoS shows extreme forward rate with near-zero backward.
- **Inter-arrival times (IAT):** min/max/mean/std of gaps between packets, plus Flow Duration. Captures *timing* — scans and floods have unusually small or regular IATs.
- **TCP flags:** SYN/FIN/RST/PSH/ACK/URG/ECE counts, initial TCP window sizes. Captures *connection behaviour* — SYN without ACK = SYN flood; window size fingerprints OS and botnet clients.
- **Other:** destination port, header lengths, active/idle time stats. Port alone is a strong discriminator — scanning generates traffic across unusual high ports.

**Labels:** 15 attack types + BENIGN in train; 10 attack types + BENIGN in test (2018 used integer encoding; 2017 had UTF-8 artifacts in Web Attack labels — both fixed at load time).

**Timestamp excluded from features:** Including raw calendar timestamps would cause time leakage ('2017 flow = benign, 2018 flow = attack'). IAT and Duration capture structural flow timing, not calendar position.


### Temporal structure and the progressive evaluation design

In [ ]:
import os, glob

# ── Per-day file breakdown for CIC-IDS2017 ────────────────────────────────────
cic2017_dir = os.path.join(DATA_DIR, 'cic2017')
cic2018_dir = os.path.join(DATA_DIR, 'cic2018')

print("CIC-IDS2017 source files (training set):")
for f in sorted(glob.glob(os.path.join(cic2017_dir, '*.csv'))):
    print(f"  {os.path.basename(f)}")

print("\nCSE-CIC-IDS2018 source files (progressive test set):")
for f in sorted(glob.glob(os.path.join(cic2018_dir, '*.csv'))):
    print(f"  {os.path.basename(f)}")

In [ ]:
# ── Attack type label distribution in each dataset ───────────────────────────
print("Attack types in TRAINING set (CIC-IDS2017):")
print(df_train['Label'].value_counts().to_string())

print("\nAttack types in TEST set (CSE-CIC-IDS2018):")
print(df_test['Label'].value_counts().to_string())

**Temporal structure — what time means in this project:**

**1. Within each dataset (daily granularity):**
CIC-IDS2017 is split across 8 CSV files covering five working days (Monday 3 July – Friday 7 July 2017). Monday is benign-only background traffic. From Tuesday onward, increasingly complex attacks are injected: Tuesday has brute-force (FTP-Patator, SSH-Patator); Wednesday has DoS/DDoS; Thursday has Web Attacks and Infiltration; Friday has DDoS and PortScan. CSE-CIC-IDS2018 arrives as a single pre-processed file — the within-day structure has been collapsed.

**2. Between datasets — the progressive gap (~8 months):**
All 2017 data precedes all 2018 data. There is zero temporal overlap. This is the core of the paper's methodology: a model trained on July 2017 traffic is evaluated on February–March 2018 traffic it has never seen.

**Class imbalance in both datasets:**
- Train: 226,117 BENIGN / 55,317 attacks → **80.4% benign**
- Test: 96,421 BENIGN / 28,475 attacks → **77.2% benign**

Both datasets reflect realistic network conditions where benign traffic dominates. This imbalance is a central methodological issue — the authors address it by downsampling to a 1:1 ratio before training.

**Attack type label distribution:**
Both datasets contain a mix of DoS, DDoS, brute-force, and bot traffic alongside the dominant BENIGN class. The two datasets use different naming conventions for some attack families (e.g. "DoS Hulk" in 2017 vs "DoS - Hulk" in 2018) — an artifact of how each dataset was labelled at collection time. For this project, all attacks are collapsed into a single binary ATTACK label for training and evaluation.

### Data hygiene scan

In [ ]:
# ── Duplicate rows ────────────────────────────────────────────────────────────
feat_cols = [c for c in df_train.columns if c != 'Label']

train_dups = df_train.duplicated(subset=feat_cols).sum()
test_dups  = df_test.duplicated(subset=feat_cols).sum()
print(f"Duplicate feature rows — train: {train_dups:,}  |  test: {test_dups:,}")

# ── Remaining NaN / inf (should be zero after _clean) ────────────────────────
train_nan = df_train[feat_cols].isnull().sum().sum()
test_nan  = df_test[feat_cols].isnull().sum().sum()
print(f"Remaining NaN values  — train: {train_nan}  |  test: {test_nan}")

train_inf = np.isinf(df_train[feat_cols].values).sum()
test_inf  = np.isinf(df_test[feat_cols].values).sum()
print(f"Remaining inf values  — train: {train_inf}  |  test: {test_inf}")

In [ ]:
# ── Constant / near-constant features (single unique value = useless) ─────────
constant_train = [c for c in feat_cols if df_train[c].nunique() <= 1]
constant_test  = [c for c in feat_cols if df_test[c].nunique()  <= 1]
print(f"Constant features in train: {constant_train or 'none'}")
print(f"Constant features in test:  {constant_test  or 'none'}")

# Near-constant: >99.9% of values are the same
near_const_train = [c for c in feat_cols
                    if df_train[c].value_counts(normalize=True).iloc[0] > 0.999]
print(f"\nNear-constant features in train (>99.9% one value): {near_const_train or 'none'}")

In [ ]:
# ── Drop duplicates and constant features; log decisions ─────────────────────
cols_to_drop = list(set(constant_train + constant_test))

df_train_clean = df_train.drop_duplicates(subset=feat_cols).reset_index(drop=True)
df_test_clean  = df_test.drop_duplicates(subset=feat_cols).reset_index(drop=True)

if cols_to_drop:
    df_train_clean = df_train_clean.drop(columns=cols_to_drop)
    df_test_clean  = df_test_clean.drop(columns=cols_to_drop)

print(f"After deduplication:")
print(f"  Train: {len(df_train):,} → {len(df_train_clean):,} rows  "
      f"(removed {len(df_train) - len(df_train_clean):,} duplicates)")
print(f"  Test:  {len(df_test):,}  → {len(df_test_clean):,}  rows  "
      f"(removed {len(df_test) - len(df_test_clean):,} duplicates)")
if cols_to_drop:
    print(f"\nDropped constant columns: {cols_to_drop}")
else:
    print("\nNo constant columns dropped.")

**Hygiene findings — cleaned shapes: 266,739 × 67 train, 106,906 × 67 test:**

- **Duplicates removed:** 14,695 from train (5.2%), 17,790 from test (14.3%). Common in network data — attack scripts hitting identical parameters, or benign apps opening repeated identical connections. Higher rate in test reflects the more uniform pre-processed 2018 file.
- **10 constant columns dropped:** `Bwd Avg Bulk Rate/Bytes/Packets`, `Fwd Avg Bulk Rate/Bytes/Bulk`, `Bwd PSH Flags`, `Bwd URG Flags`, `Fwd URG Flags`, `CWE Flag Count` — all zero across every row. Known CICFlowMeter limitation: bulk-transfer heuristics rarely trigger in lab-simulated traffic. Zero-variance features add only noise to a model.
- **Near-constant but kept:** `ECE Flag Count` and `RST Flag Count` exceed the 99.9% threshold in training but vary in the test set — rare non-zero values may still carry signal.
- Cleaned frames saved as `train_clean.joblib` / `test_clean.joblib` to Drive.


In [ ]:
# ── Save cleaned frames to Drive ──────────────────────────────────────────────
from joblib import dump

dump(df_train_clean, os.path.join(DATA_DIR, 'train_clean.joblib'))
dump(df_test_clean,  os.path.join(DATA_DIR, 'test_clean.joblib'))
print("Saved train_clean.joblib and test_clean.joblib to Drive.")

---
## §2 — Exploratory Data Analysis

Understand training and test distributions before feature engineering. Covers:
- **§2.1** Class distribution and the class-imbalance problem
- **§2.2** Feature distributions for key network-flow statistics
- **§2.3** Missing values verification
- **§2.4** Outlier analysis
- **§2.5** Temporal-feature analysis
- **§2.6** Cross-tabulation and group-by analysis
- **§2.7** Correlation analysis — method choice and justification (Spearman)
- **§2.8** Pre- vs post-balancing: visualising the 'symmetry' trade-off


In [ ]:
# §2 setup — reload cleaned frames if needed, define FIGURES_DIR
import os, pathlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from joblib import load as jload

try:
    df_train_clean
except NameError:
    df_train_clean = jload(os.path.join(DATA_DIR, 'train_clean.joblib'))
    df_test_clean  = jload(os.path.join(DATA_DIR, 'test_clean.joblib'))
    print('Reloaded cleaned frames from Drive.')

feat_cols_clean = [c for c in df_train_clean.columns if c != 'Label']
mask_benign = df_train_clean['Label'].str.strip().str.upper() == 'BENIGN'
print(f'Train: {df_train_clean.shape}, Test: {df_test_clean.shape}, Features: {len(feat_cols_clean)}')

FIGURES_DIR = str(pathlib.Path(DATA_DIR).parent / 'figures')
os.makedirs(FIGURES_DIR, exist_ok=True)
sns.set_theme(style='whitegrid', palette='tab10')


### §2.1 — Class Distribution and Imbalance Analysis


In [ ]:
def binary_counts(df):
    b = df['Label'].apply(lambda x: 'BENIGN' if str(x).strip().upper()=='BENIGN' else 'ATTACK')
    return b.value_counts().reindex(['BENIGN','ATTACK'], fill_value=0)

train_bc = binary_counts(df_train_clean)
test_bc  = binary_counts(df_test_clean)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, (name, bc) in zip(axes, [('Train (CIC-IDS2017)', train_bc),
                                   ('Test (CSE-CIC-IDS2018)', test_bc)]):
    bars = ax.bar(bc.index, bc.values, color=['steelblue','tomato'],
                  edgecolor='white', linewidth=1.2)
    ax.set_title(name, fontsize=12)
    ax.set_ylabel('Row count')
    for bar, (lbl, val) in zip(bars, bc.items()):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+300,
                f'{val:,}\n({val/bc.sum()*100:.1f}%)',
                ha='center', va='bottom', fontsize=10)

plt.suptitle('Class distribution — before balancing (real prevalence)', fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'class_distribution_raw.png'), dpi=150, bbox_inches='tight')
plt.show()

print('\n--- Train: attack-type breakdown ---')
print(df_train_clean['Label'].value_counts().to_string())
print('\n--- Test: attack-type breakdown ---')
print(df_test_clean['Label'].value_counts().to_string())


**Class distribution** *(Figure 1)*: 218,453 BENIGN vs 48,286 attack in train (**81.9% benign**); 89,937 vs 16,969 in test (**84.1% benign**). Both reflect typical enterprise network conditions.

**Attack-type drift — the core threat to the paper's conclusions:**

| Attack type | Train | Test | Change |
|-------------|-------|------|--------|
| Web Attack - XSS | 70 | Brute Force - XSS: **10,489** | ~150× amplification |
| PortScan | **14,444** | 0 | Disappears entirely |
| DDoS (generic) | 12,732 | HOIC: 3,415 / LOIC-UDP: 861 / LOIC-HTTP: 216 | Renamed & fragmented |
| Bot | 199 | 34 | Shrinks |
| Heartbleed | **1** | 0 | Effectively unlearnable |
| Infiltration | **6** | 0 | Effectively unlearnable |

A model trained on 70 XSS flows that then faces 10,489 XSS flows at test time is experiencing **concept drift**, not classical overfitting. The paper conflates the two — Phase 8.1 provides a direct test.

The authors' 1:1 downsampling also means accuracy on the balanced test set does not reflect deployment where 84% of inputs are benign. Quantified in §2.8.


### §2.2 — Feature Distributions


In [ ]:
KEY_FEATURES = [
    'Flow Duration', 'Total Fwd Packets', 'Total Backward Packets',
    'Flow Bytes/s', 'Fwd Packet Length Mean', 'Bwd Packet Length Mean',
    'Fwd IAT Mean', 'Bwd IAT Mean', 'Flow IAT Mean',
]
KEY_FEATURES = [f for f in KEY_FEATURES if f in feat_cols_clean]

ncols = 3
nrows = (len(KEY_FEATURES) + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(16, 4 * nrows))
axes = axes.flatten()

for ax, feat in zip(axes, KEY_FEATURES):
    clip_val = df_train_clean[feat].quantile(0.99)
    for lbl, mask, color in [
        ('BENIGN', mask_benign, 'steelblue'),
        ('ATTACK', ~mask_benign, 'tomato')
    ]:
        ax.hist(df_train_clean.loc[mask, feat].clip(upper=clip_val),
                bins=50, alpha=0.5, color=color, label=lbl, density=True)
    ax.set_title(feat, fontsize=9)
    ax.tick_params(labelsize=7)

axes[0].legend(fontsize=9)
for ax in axes[len(KEY_FEATURES):]:
    ax.set_visible(False)

plt.suptitle('Feature distributions by class — train set (99th-percentile clipped)', fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'feature_distributions.png'), dpi=150, bbox_inches='tight')
plt.show()


**Feature distributions** *(Figure 2)*:

- All features are **heavily right-skewed** — most flows are short with few packets, with a heavy tail of high-volume flows. This violates Pearson's normality assumption and motivates Spearman (§2.7).
- **`Flow Duration`** (attack) is bimodal: spike near zero (fast DoS floods) + mass at ~10⁸ µs (slowloris/Hulk keeping connections open). Two completely different attack mechanics in one binary label.
- **`Bwd Packet Length Mean`** (attack) is also bimodal: spike at 0 (Slowhttptest — no server response) + mass at 1,500–2,000 bytes (DDoS response packets).
- **`Flow Bytes/s`** shows negative values — a CICFlowMeter edge-case artifact in this dataset.
- The per-feature overlap between benign and attack explains why no single feature suffices; the paper's 11-feature selection (§3) targets the most discriminative *combination*.


### §2.3 — Missing Values Verification


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

for ax, (name, df) in zip(axes, [('Train', df_train_clean), ('Test', df_test_clean)]):
    nan_counts = df[feat_cols_clean].isnull().sum()
    nan_nz = nan_counts[nan_counts > 0]
    if nan_nz.empty:
        ax.text(0.5, 0.5, 'No missing values\n(cleaned in §1)',
                ha='center', va='center', transform=ax.transAxes,
                fontsize=13, color='seagreen')
    else:
        nan_nz.sort_values(ascending=False).plot(kind='bar', ax=ax, color='tomato')
        ax.set_ylabel('NaN count')
    ax.set_title(f'{name} — NaN per feature')

plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'missing_values.png'), dpi=150, bbox_inches='tight')
plt.show()

for name, df in [('Train', df_train_clean), ('Test', df_test_clean)]:
    numeric = df[feat_cols_clean].select_dtypes(include='number')
    inf_count = np.isinf(numeric).sum().sum()
    print(f'{name} — inf values remaining: {inf_count}')


### §2.4 — Outlier Analysis


In [ ]:
print('Outlier rate (IQR method, >Q3 + 1.5*IQR) — train set:\n')
outlier_summary = {}
for feat in KEY_FEATURES:
    q1, q3 = df_train_clean[feat].quantile([0.25, 0.75])
    upper = q3 + 1.5 * (q3 - q1)
    n_out = int((df_train_clean[feat] > upper).sum())
    outlier_summary[feat] = n_out
    print(f'  {feat:<35}: {n_out:>6,}  ({n_out/len(df_train_clean)*100:.1f}%)')

top6 = sorted(outlier_summary, key=outlier_summary.get, reverse=True)[:6]
top6 = [f for f in top6 if f in df_train_clean.columns]

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, feat in zip(axes.flatten(), top6):
    cap = df_train_clean[feat].quantile(0.999)
    data = [
        df_train_clean.loc[mask_benign, feat].clip(upper=cap).values,
        df_train_clean.loc[~mask_benign, feat].clip(upper=cap).values,
    ]
    bp = ax.boxplot(data, tick_labels=['Benign', 'Attack'], patch_artist=True, showfliers=False)
    for patch, color in zip(bp['boxes'], ['steelblue', 'tomato']):
        patch.set_facecolor(color)
        patch.set_alpha(0.6)
    ax.set_title(feat, fontsize=9)
    ax.tick_params(labelsize=8)

plt.suptitle(
    'Box plots — highest-outlier features by class (99.9th-pct clipped, no fliers shown)',
    fontsize=11
)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'outlier_boxplots.png'), dpi=150, bbox_inches='tight')
plt.show()


**Outlier findings** *(Figure 4)*: IQR outlier rates of 5–20% per feature — expected, not a quality problem.

- **DoS/DDoS:** outliers in `Flow Bytes/s` and IAT — flooding at maximum rate.
- **PortScan:** extreme values in packet-length features (tiny probes) and very short `Flow Duration`.
- **Benign elephant flows:** genuine outliers in byte-count features (large file transfers, video streaming).
- **Counterintuitive:** BENIGN has a *higher* median `Flow Bytes/s` than ATTACK (~210k vs ~100k bytes/s). Slow DoS attacks (slowloris, Slowhttptest) have near-zero byte rates, pulling the attack median down. `Flow Bytes/s` alone is not a reliable attack indicator.

**Decision — do not clip outliers:** tree models (DT, RF) handle them via split thresholds; SVM/ANN inputs will be scaled in §3. Clipping would destroy the attack-fingerprint patterns.


### §2.5 — Temporal Feature Analysis


In [ ]:
TEMPORAL_FEATS = [f for f in
    ['Flow Duration', 'Fwd IAT Mean', 'Bwd IAT Mean', 'Flow IAT Mean']
    if f in feat_cols_clean]

top10_labels = df_train_clean['Label'].value_counts().head(10).index.tolist()
df_sub = df_train_clean[df_train_clean['Label'].isin(top10_labels)]

fig, axes = plt.subplots(len(TEMPORAL_FEATS), 1, figsize=(13, 4 * len(TEMPORAL_FEATS)))
if len(TEMPORAL_FEATS) == 1:
    axes = [axes]

for ax, feat in zip(axes, TEMPORAL_FEATS):
    medians = df_sub.groupby('Label')[feat].median().sort_values()
    colors = ['steelblue' if 'BENIGN' in str(l).upper() else 'tomato'
              for l in medians.index]
    ax.barh(medians.index, medians.values, color=colors)
    ax.set_title(f'Median {feat} by class', fontsize=11)
    ax.set_xlabel(feat)
    ax.tick_params(labelsize=9)

plt.suptitle('Temporal features by attack class — top 10 classes, train set', fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'temporal_features_by_class.png'), dpi=150, bbox_inches='tight')
plt.show()


**Temporal features by class** *(Figure 5)* — attack fingerprints align with known network behaviour:

- **DoS/DDoS** (GoldenEye, Hulk, LOIC/HOIC): near-zero IAT + short duration — flooding at maximum rate.
- **Slowloris / Slowhttptest:** longest flow durations (~10⁸ µs) — intentionally kept alive; near-zero Bwd IAT because server barely responds.
- **DoS Hulk:** high Fwd IAT despite being a DoS attack — Hulk sends valid HTTP GETs with pauses, not raw packet flooding.
- **PortScan:** near-zero duration and IAT (connection attempts fail immediately).
- **FTP-Patator / SSH-Patator:** moderate duration (full authentication protocol exchange per attempt).
- **BENIGN:** highest IAT variance — irregular, human-paced behaviour mixing DNS queries and streaming.

Note: DoS attacks dominate the axis scale — BENIGN, Bot, and PortScan bars are nearly invisible in Figure 5. This is a scale limitation, not a data problem.

**Timestamp excluded:** including calendar timestamps would teach the model '2017 = benign, 2018 = attack' (time leakage). IAT and Duration capture structural flow timing and are safe.


### §2.6 — Cross-tabulation and Group-by Analysis


In [ ]:
SUMMARY_FEATS = [f for f in
    ['Flow Duration','Total Fwd Packets','Total Backward Packets',
     'Flow Bytes/s','Fwd Packet Length Mean','Bwd Packet Length Mean']
    if f in feat_cols_clean]

top8 = df_train_clean['Label'].value_counts().head(8).index.tolist()
group_stats = (
    df_train_clean[df_train_clean['Label'].isin(top8)]
    .groupby('Label')[SUMMARY_FEATS]
    .median()
    .round(2)
)
print('Median feature values by class (top 8, train set):')
print(group_stats.to_string())
print()

if 'Total Fwd Packets' in feat_cols_clean and 'Total Backward Packets' in feat_cols_clean:
    ratio = (df_train_clean['Total Backward Packets'] /
             (df_train_clean['Total Fwd Packets'] + 1e-6)).clip(0, 100)
    ratio_by_class = (
        df_train_clean[df_train_clean['Label'].isin(top8)]
        .assign(_ratio=ratio)
        .groupby('Label')['_ratio']
        .median()
        .sort_values()
    )
    print('Median Bwd/Fwd packet ratio by class:')
    print('  (0 = purely unidirectional DoS;  ~1 = balanced bidirectional traffic)')
    print(ratio_by_class.to_string())


**Bwd/Fwd packet ratio by class** — confirms known network-security patterns:

| Class | Median Bwd/Fwd | Note |
|-------|---------------|------|
| DoS Slowhttptest | **0.000** | Purely unidirectional — server never responds |
| DoS slowloris | 0.214 | Mostly unidirectional — connection starved |
| DoS GoldenEye | 0.625 | HTTP requests get some responses before slowdown |
| DDoS | 0.750 | Victim responds to a fraction of the flood |
| DoS Hulk | 0.857 | Near-symmetric — server tries to answer valid HTTP GETs |
| BENIGN | **1.000** | Perfectly bidirectional (TCP/HTTP/DNS) |
| PortScan | **1.000** | CICFlowMeter captures SYN+SYN-ACK → appears symmetric |
| FTP-Patator | **1.667** | *Server-heavy:* multiple 530 failure messages per login attempt |

- **FTP-Patator > 1** is the most unusual finding: attacker-initiated traffic generates *more server-side* packets than client-side — the FTP server sends banners, challenges, and error codes for every failed attempt.
- **PortScan = 1** is counterintuitive: CICFlowMeter captures completed SYN+SYN-ACK flows, not raw SYN probes — so packet-direction features alone won't easily distinguish scans from benign traffic.


### §2.7 — Correlation Analysis — Method Choice and Justification


**Why Spearman (not Pearson or Kendall):**

| Method | Key assumption | Verdict for this dataset |
|--------|---------------|-------------------------|
| **Pearson** | Linear relationship; normally distributed, homoscedastic data | ❌ Network features follow power-law distributions; outliers are attack signals, not errors; relationship is monotonic but not strictly linear |
| **Kendall** | Rank-based; no normality assumption; robust to outliers | ✅ Correct assumptions, but **O(n²)** computation — impractical for ~267k rows and 66 features |
| **Spearman** | Rank-based; no normality assumption; robust to outliers | ✅ Captures monotonic relationships; **O(n log n)**; correct and efficient for this scale |

**Practical vs. statistical significance:** With ~267k rows, virtually every non-zero correlation is statistically significant (p < 0.001). What matters is *practical* significance: |r| > 0.90 identifies feature pairs carrying essentially redundant information — candidates for removal in §3 feature selection.


In [ ]:
# Spearman on a 10k-row subsample — fast and representative
_CORR_N = 10_000
df_corr_sample = df_train_clean[feat_cols_clean].sample(
    min(_CORR_N, len(df_train_clean)), random_state=SEED
)
print(f'Computing Spearman matrix on {len(df_corr_sample):,} rows x {len(feat_cols_clean)} features...')
corr_matrix = df_corr_sample.corr(method='spearman')
print('Done.')

# Pairs with |r| > 0.90 — redundant features
high_pairs = []
cols = corr_matrix.columns.tolist()
for i in range(len(cols)):
    for j in range(i+1, len(cols)):
        r = float(corr_matrix.iloc[i, j])
        if abs(r) > 0.90:
            high_pairs.append((abs(r), r, cols[i], cols[j]))
high_pairs.sort(reverse=True)

print(f'\nFeature pairs |Spearman r| > 0.90  ({len(high_pairs)} total — top 20 shown):')
for _, r, c1, c2 in high_pairs[:20]:
    print(f'  r={r:+.3f}  {c1}  <->  {c2}')


In [ ]:
# Spearman heatmap — top 30 highest-variance features, hierarchically clustered
from scipy.cluster.hierarchy import linkage, leaves_list
from scipy.spatial.distance import squareform

top30 = df_corr_sample.var().nlargest(30).index.tolist()
sub_corr = corr_matrix.loc[top30, top30]

try:
    dist_mat = (1 - sub_corr.abs()).clip(lower=0)
    np.fill_diagonal(dist_mat.values, 0.0)
    link = linkage(squareform(dist_mat.values), method='average')
    order = leaves_list(link)
    sub_corr = sub_corr.iloc[order, order]
except Exception as e:
    print(f'Hierarchical clustering skipped ({e}); using original order.')

fig, ax = plt.subplots(figsize=(14, 12))
sns.heatmap(sub_corr, ax=ax, cmap='RdBu_r', center=0, vmin=-1, vmax=1,
            square=True, linewidths=0.3, cbar_kws={'label': 'Spearman r'},
            xticklabels=True, yticklabels=True)
ax.tick_params(axis='x', labelrotation=45, labelsize=7)
ax.tick_params(axis='y', labelrotation=0, labelsize=7)
ax.set_title('Spearman correlation — top 30 highest-variance features (train set)', fontsize=12)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'spearman_correlation_heatmap.png'), dpi=150, bbox_inches='tight')
plt.show()


**Spearman correlation** *(Figure 6)*: 99 pairs with |r| > 0.90. Ten pairs are r = 1.000 (mathematically identical):

| Pair (r = 1.000) | Reason |
|-----------------|--------|
| `Subflow Fwd/Bwd Packets/Bytes` ↔ `Total Fwd/Bwd Packets/Length` (4 pairs) | CICFlowMeter: Subflow = Total for single-subflow flows |
| `Avg Fwd/Bwd Segment Size` ↔ `Fwd/Bwd Packet Length Mean` (2 pairs) | Two names for the same computation |
| `Packet Length Std` ↔ `Packet Length Variance` | Mathematical identity: Var = Std² |
| `Idle Max` ↔ `Idle Mean` | Near-constant idle times |
| `Fwd PSH Flags` ↔ `SYN Flag Count` | CIC capture artefact |
| `ECE Flag Count` ↔ `RST Flag Count` | CIC capture artefact |

**Three redundancy clusters (visible in Figure 6):**
1. **Subflow cluster:** four subflow features are exact duplicates of four total-flow features → all four should be dropped.
2. **Packet length / segment size:** `Avg Segment Size` = `Packet Length Mean`; `Packet Length Variance` = `Packet Length Std²` → keep one per pair.
3. **IAT cluster:** `Bwd/Fwd/Flow IAT Max/Mean/Total` all r ≥ 0.997 with each other → one representative suffices.

The flag artefacts (PSH↔SYN, ECE↔RST) are CIC-specific — if these correlations break in a different network capture, a model relying on them will degrade, contributing to the cross-dataset performance drop.

This directly motivates the authors' **66 → 11 feature reduction** reproduced in §3.


### §2.8 — Pre- vs Post-Balancing: the 'Symmetry' Trade-off


In [ ]:
df_bin = df_train_clean.assign(
    _binary=df_train_clean['Label'].apply(
        lambda x: 'BENIGN' if str(x).strip().upper()=='BENIGN' else 'ATTACK'
    )
)
before = df_bin['_binary'].value_counts().reindex(['BENIGN','ATTACK'], fill_value=0)

n_min = int(before.min())
balanced = pd.concat([
    df_bin[df_bin['_binary']==cls].sample(n=n_min, random_state=SEED)
    for cls in ['BENIGN','ATTACK']
])
after = balanced['_binary'].value_counts().reindex(['BENIGN','ATTACK'])

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, (title, bc) in zip(axes, [
    ('Before balancing — real prevalence', before),
    ("After 1:1 downsampling (paper's 'symmetry')", after)
]):
    bars = ax.bar(bc.index, bc.values, color=['steelblue','tomato'],
                  edgecolor='white', linewidth=1.2)
    ax.set_title(title, fontsize=11)
    ax.set_ylabel('Row count')
    total = bc.sum()
    for bar, val in zip(bars, bc.values):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+200,
                f'{val:,}\n({val/total*100:.1f}%)',
                ha='center', va='bottom', fontsize=10)

plt.suptitle("Class distribution: real prevalence vs. paper's 1:1 'symmetry'", fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'class_balancing_effect.png'), dpi=150, bbox_inches='tight')
plt.show()

discarded = int(before.sum() - after.sum())
print(f'Rows discarded by 1:1 downsampling: {discarded:,}')
print(f'Training data retained: {after.sum()/before.sum()*100:.1f}%')


**The real cost of 1:1 'symmetry'** *(Figure 7)*: keeping only 48,286 of 218,453 BENIGN rows means **170,167 rows discarded — 63.8% of all training data**. Only 36.2% is used.

- **Metric inflation:** accuracy on a 1:1 balanced test set is systematically higher than on a deployment-realistic 84%-benign set. The paper's reported ~96–99% in-distribution accuracy is an upper bound, not a deployment estimate.
- **Data waste:** the model sees a less representative sample of legitimate traffic — rare-but-legitimate flows that fall in the discarded majority may become false positives.
- **Alternatives not explored:** class-weighted loss, SMOTE (synthetic oversampling), or threshold tuning — all address imbalance without discarding 63.8% of the data. The paper's 'symmetry' framing elevates a pragmatic compromise to a principle it doesn't warrant.

Phase 8.2 re-evaluates all models on the real-prevalence test split to measure the accuracy gap directly.


---
## §3 — Feature Engineering

Reproduce the authors' preprocessing pipeline, then add feature creation,
scaling, feature selection (RF importance + brute-force add-one loop),
and a redundancy analysis. Steps:

- **§3.1** Cleaning recap — mirrors the authors' `01_Dataset_Preprocessing.ipynb`
- **§3.2** Binary relabelling — collapse 15 attack types → `ATTACK`
- **§3.3** Class balancing — 1:1 downsample (paper's method) + preserve real-prevalence copy
- **§3.4** Categorical encoding — choice and justification
- **§3.5** Derived features — four engineered flow metrics with cyber rationale
- **§3.6** Feature scaling — StandardScaler fitted on balanced training set
- **§3.7** Feature selection — RF importance ranking + brute-force add-one accuracy curve
- **§3.8** Redundancy analysis — correlation clusters, RF importance, VIF


In [ ]:
# §3 setup — reload cleaned frames if needed; import preprocessing module
import os, pathlib, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from joblib import dump, load as jload

try:
    df_train_clean
except NameError:
    df_train_clean = jload(os.path.join(DATA_DIR, 'train_clean.joblib'))
    df_test_clean  = jload(os.path.join(DATA_DIR, 'test_clean.joblib'))
    print('Reloaded cleaned frames from Drive.')

from src.preprocessing import (
    binary_relabel, balance_1to1, add_derived_features, get_feature_cols,
    fit_scaler, apply_scaler, select_features_rf, brute_force_select, compute_vif,
)

FIGURES_DIR = str(pathlib.Path(DATA_DIR).parent / 'figures')
os.makedirs(FIGURES_DIR, exist_ok=True)
sns.set_theme(style='whitegrid', palette='tab10')

print(f'Train: {df_train_clean.shape}  |  Test: {df_test_clean.shape}')
print(f'Features available: {len(get_feature_cols(df_train_clean))}')


### §3.1 — Cleaning Recap — mirroring the authors


In [ ]:
# §3.1 — Cleaning was performed in §1; document it here for the reproducibility record.
print('=== Cleaning steps applied in §1 (mirrors 01_Dataset_Preprocessing.ipynb) ===\n')
steps = [
    ('1. Strip column-name whitespace',
     'Identical to authors — CICFlowMeter produces inconsistent spacing'),
    ('2. Drop Fwd Header Length.1',
     'Duplicate of Fwd Header Length; present only in CIC-IDS2017'),
    ('3. Replace ±inf → NaN, dropna()',
     'Identical to authors — removes ~0.05% train rows, ~0.63% test rows'),
    ('4. Downcast float64 → float32',
     'Our addition — halves RAM on free Colab; negligible precision loss'),
    ('5. Drop duplicate feature rows',
     'Our addition — removes 5.2% train / 14.3% test; reduces noise from repeated flows'),
    ('6. Drop 10 constant columns',
     'Our addition — zero-variance features (CICFlowMeter bulk-rate artefacts)'),
]
for step, note in steps:
    print(f'  {step}\n    → {note}\n')
print(f'Resulting shapes:  train {df_train_clean.shape}  |  test {df_test_clean.shape}')
print(f'Features: {len(get_feature_cols(df_train_clean))}  (down from 78 in raw CSVs)')


**Cleaning decisions match the authors** for the core three steps (strip names, drop duplicate column,
remove inf/NaN). Two extra steps were added — `float32` downcasting and deduplication — to fit
free Colab RAM and reduce noise. Neither changes statistical structure: deduplication removes
exact-copy rows (no additional information), and float32 rounding error is below 1e-7.


### §3.2 — Binary Relabelling


In [ ]:
# §3.2 — Collapse 15+ attack types → binary ATTACK / BENIGN.
# Mirrors the authors' 01_Dataset_Preprocessing.ipynb Step 4.
df_train_bin = binary_relabel(df_train_clean)
df_test_bin  = binary_relabel(df_test_clean)

for name, df_orig, df_bin in [
    ('Train', df_train_clean, df_train_bin),
    ('Test',  df_test_clean,  df_test_bin),
]:
    orig_classes = df_orig['Label'].nunique()
    vc = df_bin['Label'].value_counts()
    print(f'{name}: {orig_classes} original classes → 2 binary classes')
    print(f'  BENIGN : {vc["BENIGN"]:>7,}  ({vc["BENIGN"]/len(df_bin)*100:.1f}%)')
    print(f'  ATTACK : {vc["ATTACK"]:>7,}  ({vc["ATTACK"]/len(df_bin)*100:.1f}%)')
    print()


**Binary relabelling rationale:** The paper collapses all attack types to a single `malicious` label
to focus on the binary detection task. This simplifies the target and allows a fair 6-model comparison
without multiclass complications.

**Trade-off:** binary collapse discards within-class structure. As shown in §2.1, XSS in train (n=70)
and XSS in test (n=10,489) are statistically very different — binary collapse hides this distribution shift.
The original multiclass `Label` is preserved in `df_train_clean` / `df_test_clean`
for the optional multiclass extension (Phase 8.3).


### §3.3 — Class Balancing (1:1) and Real-Prevalence Copy


In [ ]:
# §3.3 — Downsample BENIGN to 1:1; keep a real-prevalence copy for Phase 8.2.
# Only the training set is balanced; the test set stays at real prevalence.
df_train_bal, df_train_imbal = balance_1to1(df_train_bin, seed=SEED)

print('=== Training set: before and after 1:1 balancing ===')
for label, df in [('Real prevalence (Phase 8.2 copy)', df_train_imbal),
                   ('1:1 balanced    (mirrors paper)',  df_train_bal)]:
    vc = df['Label'].value_counts()
    total = len(df)
    print(f'\n  {label}')
    print(f'    BENIGN: {vc["BENIGN"]:>7,}  ({vc["BENIGN"]/total*100:.1f}%)')
    print(f'    ATTACK: {vc["ATTACK"]:>7,}  ({vc["ATTACK"]/total*100:.1f}%)')
    print(f'    Total:  {total:>7,}')

discarded = len(df_train_imbal) - len(df_train_bal)
print(f'\nRows discarded: {discarded:,} ({discarded/len(df_train_imbal)*100:.1f}% of training data)')
print('Test set: kept at real prevalence — not balanced.')


**Two copies kept deliberately:**

- `df_train_bal` — 1:1 balanced copy used for all model training and in-distribution
  evaluation (mirrors the paper). The 50/50 ratio means a trivial classifier achieves 50% accuracy,
  so any model above that is doing real work.
- `df_train_imbal` — real-prevalence copy (~82% BENIGN) used in Phase 8.2 to re-score
  models under deployment conditions and challenge the paper's "symmetry" framing.

**Critique note (→ report §2):** The paper discards ~63.8% of training data to achieve 1:1.
Alternatives — `class_weight='balanced'`, SMOTE oversampling, or decision-threshold tuning —
would address imbalance without discarding majority-class information.
The paper does not compare these alternatives.


### §3.4 — Categorical Encoding — Choice and Justification


In [ ]:
# §3.4 — Check for non-numeric feature columns; discuss Destination Port encoding.
feat_cols = get_feature_cols(df_train_bal)

dtype_counts = df_train_bal[feat_cols].dtypes.value_counts()
print('Feature dtype breakdown (balanced training set):')
print(dtype_counts.to_string())

non_numeric = df_train_bal[feat_cols].select_dtypes(exclude='number').columns.tolist()
print(f'\nNon-numeric feature columns: {non_numeric or "none — all features are numeric"}')

# Destination Port — the integer feature that warrants encoding discussion
port_col = 'Destination Port'
if port_col in df_train_bal.columns:
    port = df_train_bal[port_col]
    print(f'\nDestination Port: dtype={port.dtype}, unique={port.nunique():,}, '
          f'min={port.min():.0f}, max={port.max():.0f}')
    print(f'Top-10 most frequent ports in balanced train:')
    print(port.value_counts().head(10).to_dict())
    well_known = (port < 1024).sum()
    registered = ((port >= 1024) & (port < 49152)).sum()
    dynamic    = (port >= 49152).sum()
    print(f'\nPort range breakdown:  well-known 0-1023: {well_known:,}  '
          f'registered 1024-49151: {registered:,}  dynamic 49152+: {dynamic:,}')


**Encoding decision: treat `Destination Port` as a continuous numeric integer (no encoding applied).**

All 66 features are already numeric after §1 cleaning. The only encoding question is `Destination Port`:

| Option | Problem |
|--------|---------|
| **One-hot** | Up to 65,536 unique ports → impractical sparse columns |
| **Label/ordinal** | Arbitrary ordering (port 443 is not numerically "larger" than port 80 in a meaningful sense) |
| **Port-range bins** | Loses within-bin discrimination (port 22 vs port 80 collapse to one category) |
| **Keep as integer (chosen)** | Port ranges have genuine ordinal structure; tree models learn port-specific splits (`port ≤ 1023`); SVM/ANN treat it as a feature on equal scale with others after StandardScaling |

The paper treats `Destination Port` as numeric throughout — it appears in their final 11-feature set.
**Limitation for generalisation:** port-specific patterns from CIC-IDS2017 (e.g., scans targeting
port 80/443) may not transfer to CIC-IDS2018 if attack tools target different ports —
contributing to the cross-dataset performance drop.


### §3.5 — Derived Features


In [ ]:
# §3.5 — Add four engineered flow features.
# Applied to all three sets independently (no target leakage, no cross-set statistics).
df_train_bal   = add_derived_features(df_train_bal)
df_train_imbal = add_derived_features(df_train_imbal)
df_test_bin    = add_derived_features(df_test_bin)

derived = ['feat_bwd_fwd_ratio', 'feat_pkt_len_range', 'feat_bytes_per_pkt', 'feat_win_ratio']
print('New derived feature stats (balanced train set):\n')
print(df_train_bal[derived].describe().round(3).to_string())
print('\nMedian by class (balanced train):')
print(df_train_bal.groupby('Label')[derived].median().round(3).to_string())


**Derived feature stats from the actual run** (balanced train set, medians by class):

| Feature | Median ATTACK | Median BENIGN | Discriminative? |
|---------|--------------|---------------|-----------------|
| `feat_bwd_fwd_ratio` | 1.000 | 1.000 | **No — same median for both classes** |
| `feat_pkt_len_range` | 2,896 | 54 | **Yes — 54× difference** |
| `feat_bytes_per_pkt` | 427.5 | 64.7 | **Yes — 6.6× difference** |
| `feat_win_ratio` | 123.7 | 0.235 | **Yes — 526× difference** |

**`feat_bwd_fwd_ratio` did not make the top 20 — and the median table explains why.**
In §2.6 (group-by analysis) this feature separated attack *types* clearly
(DoS Slowhttptest=0, FTP-Patator=1.667), but once all attacks are binary-collapsed,
the diverse attack types average out and the binary ATTACK median is 1.0 — identical to BENIGN.
This is a direct consequence of the paper's binary-collapse decision: per-type signals
are erased when merged into a single ATTACK class.

**`feat_win_ratio` is the standout success** — 526× median difference between classes,
ranked 3rd by RF importance (0.0595), beating almost all original features.
The TCP window size asymmetry is a strong fingerprint: attack tools (scanners, bots)
systematically use non-OS-default window sizes.

**2 of 4 derived features entered the final top-11:**
`feat_win_ratio` (rank 3) and `feat_bytes_per_pkt` (rank 11).
`feat_pkt_len_range` ranked 16th — useful but not top-11.
`feat_bwd_fwd_ratio` ranked below top-20 entirely.


### §3.6 — Feature Scaling


In [ ]:
# §3.6 — Fit StandardScaler on the balanced training set;
# apply to the imbalanced training copy and the test set.
feat_cols = get_feature_cols(df_train_bal)

X_bal   = df_train_bal[feat_cols]
y_bal   = df_train_bal['Label']
X_imbal = df_train_imbal[feat_cols]
y_imbal = df_train_imbal['Label']
X_test  = df_test_bin[feat_cols]
y_test  = df_test_bin['Label']

scaler, X_bal_scaled = fit_scaler(X_bal)
X_imbal_scaled = apply_scaler(scaler, X_imbal)
X_test_scaled  = apply_scaler(scaler, X_test)

print('Scaler fitted on balanced training set only (no leakage from imbal/test).')
print(f'X_bal_scaled   shape: {X_bal_scaled.shape}  '
      f'mean_max_abs: {X_bal_scaled.mean().abs().max():.5f}  '
      f'std_mean: {X_bal_scaled.std().mean():.5f}')
print(f'X_imbal_scaled shape: {X_imbal_scaled.shape}')
print(f'X_test_scaled  shape: {X_test_scaled.shape}')
print()
print('Sample feature means/stds on scaled training set:')
for c in feat_cols[:4]:
    print(f'  {c:<45} mean={X_bal_scaled[c].mean():+.4f}  std={X_bal_scaled[c].std():.4f}')


**Scaling rationale:**

- **SVM (RBF kernel):** distance-based — `Destination Port` (0–65 535) would dominate
  `FIN Flag Count` (0–5) without scaling, making the SVM effectively ignore low-range features.
- **ANN/DNN (MLPClassifier):** gradient descent converges faster and more reliably
  with zero-mean, unit-variance inputs.
- **DT/RF/NB:** insensitive to scale — scaling changes nothing for these models,
  but applying it uniformly simplifies the pipeline.

**Implementation note:** the scaler is fitted **only on the balanced training set**
and then applied to the imbalanced copy and the test set.
Fitting on test data would leak the test distribution into the training normalisation — a data-leakage bug.


### §3.7 — Feature Selection — RF Importance + Brute-Force Add-One


In [ ]:
# §3.7a — Stage 1: RF importance ranking on the balanced, scaled training set.
# Mirrors the authors' 02_Feature_Selection.ipynb Stage 1.
print('Fitting RF for feature importance ranking...')
t0 = time.time()
top20_features, full_importance = select_features_rf(
    X_bal_scaled, y_bal, top_n=20, seed=SEED
)
print(f'Done in {time.time()-t0:.1f}s\n')

print('Top-20 features by RF importance:')
for rank, feat in enumerate(top20_features, 1):
    imp_val = full_importance[feat]
    print(f'  {rank:2d}. {feat:<45} {imp_val:.4f}')


In [ ]:
# §3.7b — Feature importance bar chart (top 20)
fig, ax = plt.subplots(figsize=(10, 7))
imp_top20 = full_importance.head(20).sort_values(ascending=True)

palette = sns.color_palette('Blues_d', len(imp_top20))
bars = ax.barh(imp_top20.index, imp_top20.values, color=palette, edgecolor='none')

ax.set_xlabel('Mean impurity decrease (RF Gini importance)', fontsize=11)
ax.set_title('Feature importance — top 20 (RF, 100 trees, balanced train set)', fontsize=12)
ax.tick_params(axis='y', labelsize=9)
for bar, val in zip(bars, imp_top20.values):
    ax.text(val + 0.0002, bar.get_y() + bar.get_height() / 2,
            f'{val:.4f}', va='center', fontsize=8)

plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'feature_importance_rf.png'), dpi=150, bbox_inches='tight')
plt.show()


**RF importance results** *(Figure 8)* — actual top-20 from this run:

**Visual structure of the bar chart:** There is a clear cliff after the top 4 features
(Average Packet Size 0.0787, Bwd Packet Length Std 0.0627, feat_win_ratio 0.0595,
Packet Length Std 0.0587), which collectively account for ~23% of total importance.
After rank 4 the bars drop to ~0.04 and taper gradually to ~0.019 at rank 20.

**Key observations:**

- **`feat_win_ratio` ranks 3rd (0.0595)** — our derived feature beats Init_Win_bytes_forward
  (rank 20, 0.0189) and every IAT feature (all ranked 32+). TCP window asymmetry captures
  something the original features did not isolate as a single discriminant.

- **`feat_bytes_per_pkt` ranks 11th (0.0340)** — also a derived feature,
  just inside the top-11 cutoff.

- **`feat_pkt_len_range` ranks 16th (0.0234)** — useful but not top-11.

- **`feat_bwd_fwd_ratio` is below rank 20** — does not appear at all. The binary collapse
  erases the per-attack-type directionality signal identified in §2.6.

- **`Subflow Fwd Packets` appears at rank 18 (0.0206)** despite being a perfect duplicate
  of `Total Fwd Packets`. The RF splits importance mass between the two identical features.
  This is expected RF behaviour — it doesn't know the two columns are identical;
  it just discovers that each provides some split quality.

- **All IAT features rank 32+** — very low importance in binary classification.
  IAT features separated attack *types* beautifully in §2.5 (DoS=near-zero, slowloris=long)
  but once collapsed to binary, the diverse timing patterns average out.

**Comparison with the paper's top-11 features** (from paper notes):

| # | Paper's feature | Our rank | In our top-11? |
|---|----------------|----------|----------------|
| 1 | Bwd Packet Length Std | 2 | YES |
| 2 | Average Packet Size | 1 | YES |
| 3 | Max Packet Length | 12 | No |
| 4 | Packet Length Variance | 6 | YES |
| 5 | Packet Length Std | 4 | YES |
| 6 | Avg Bwd Segment Size | 9 | YES |
| 7 | Packet Length Mean | 27 | No |
| 8 | Destination Port | 5 | YES |
| 9 | Init_Win_bytes_forward | 20 | No |
|10 | Fwd Packet Length Mean | 24 | No |
|11 | Init_Win_bytes_backward | 8 | YES |

**7 of 11 features overlap.** Our 4 replacements: feat_win_ratio (rank 3), Fwd Packet Length Max
(rank 7), Bwd Packet Length Mean (rank 10), feat_bytes_per_pkt (rank 11) displaced
Max Packet Length, Packet Length Mean, Init_Win_bytes_forward, and Fwd Packet Length Mean.
Differences reflect the 10% subsample, fixed seed, and the addition of derived features.


In [ ]:
# §3.7c — Stage 2: brute-force add-one-feature accuracy curve.
# Mirrors the authors' 02_Feature_Selection.ipynb Stage 2.
# Uses LinearSVC/NaiveBayes/MLP on a 10k subsample, 3-fold CV.
# Expected runtime: ~3–5 min on Colab CPU.
print('Running brute-force feature selection (~3–5 min)...')
t0 = time.time()
bf_results = brute_force_select(
    X_bal_scaled, y_bal,
    ranked_features=top20_features,
    max_features=20,
    subsample_n=10_000,
    cv=3,
    seed=SEED,
)
print(f'\nDone in {time.time()-t0:.1f}s\n')
print(bf_results.to_string(index=False, float_format='{:.4f}'.format))


In [ ]:
# §3.7d — Accuracy vs. #features curve
fig, ax = plt.subplots(figsize=(11, 5))
color_map  = {'LinearSVC': 'steelblue', 'NaiveBayes': 'tomato', 'MLP': 'seagreen'}
marker_map = {'LinearSVC': 'o', 'NaiveBayes': 's', 'MLP': '^'}

for model in ['LinearSVC', 'NaiveBayes', 'MLP']:
    ax.plot(bf_results['n_features'], bf_results[model],
            label=model, color=color_map[model], marker=marker_map[model],
            linewidth=2, markersize=6)

ax.set_xlabel('Number of features (added in RF-importance order)', fontsize=11)
ax.set_ylabel('3-fold CV accuracy (10 k-row subsample)', fontsize=11)
ax.set_title('Feature selection — brute-force add-one accuracy curve', fontsize=12)
ax.legend(fontsize=10)
ax.set_xticks(bf_results['n_features'])
ax.grid(True, alpha=0.4)
ax.set_ylim(0.5, 1.02)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'feature_selection_curve.png'), dpi=150, bbox_inches='tight')
plt.show()

# Print plateau point for each model
for model in ['LinearSVC', 'MLP']:
    accs = bf_results[model].values
    diffs = np.diff(accs)
    plateau = [n + 2 for n in range(len(diffs)) if all(abs(d) < 0.005 for d in diffs[n:])]
    if plateau:
        print(f'{model}: plateau at n={plateau[0]}  (acc={accs[plateau[0]-1]:.4f})')


In [ ]:
# §3.7e — Finalise the selected feature set.
# Inspect bf_results + the plot above, then set OPTIMAL_N to the plateau.
# Default: 11, matching the authors. Adjust after running if your curve differs.
OPTIMAL_N = 11

selected_features = top20_features[:OPTIMAL_N]

print(f'Selected {OPTIMAL_N} features (RF importance order):')
for i, feat in enumerate(selected_features, 1):
    print(f'  {i:2d}. {feat}')

# Rebuild scaled X matrices with only the selected features
X_bal_sel   = X_bal_scaled[selected_features]
X_imbal_sel = X_imbal_scaled[selected_features]
X_test_sel  = X_test_scaled[selected_features]

print(f'\nX_bal_sel:   {X_bal_sel.shape}')
print(f'X_imbal_sel: {X_imbal_sel.shape}')
print(f'X_test_sel:  {X_test_sel.shape}')


**Feature selection curve** *(Figure 9)* — reading the actual output:

The curve is **not** a simple elbow. It has three distinct phases:

**Phase 1 — cold start (n=1–2):** All three models start low and LinearSVC actually
*drops* from n=1 to n=2 (0.699→0.684). Adding `Bwd Packet Length Std` alone (the 2nd feature)
confuses LinearSVC before it gains context from a 3rd feature. NaiveBayes also drops (0.716→0.705).

**Phase 2 — major jump at n=3:** Adding `feat_win_ratio` (rank 3) causes the single largest
accuracy gain: LinearSVC 0.684→0.868, NaiveBayes 0.705→0.850, MLP 0.857→0.881.
This confirms feat_win_ratio is the most structurally important feature *in combination*
with the top-2 packet-size features.

**Phase 3a — LinearSVC first plateau (n=7–13, ~0.882–0.883):**
LinearSVC gains almost nothing from features 7 through 13.
The 5 packet-length features (ranks 6–11) are essentially collinear — LinearSVC already
has a near-optimal hyperplane and adding correlated features doesn't help it.

**Phase 3b — NaiveBayes cliff at n=12–13:** NaiveBayes *drops* from 0.841 (n=11) to
0.723 (n=12) when `Max Packet Length` enters. This is the clearest observable consequence
of NB's independence assumption: adding a highly correlated feature causes NB to
double-count evidence, which *hurts* accuracy. NaiveBayes partially recovers at n=14 (0.841).
This behaviour foreshadows NaiveBayes's poor progressive-test performance in §5.

**Phase 4 — second jump at n=14 (LinearSVC 0.883→0.921):**
Adding `Bwd Packet Length Min` (rank 14) provides a substantial gain for LinearSVC (+3.8 pp)
and MLP (0.936→0.943). This is unexpected given these are still packet-length features.
The min-value features may capture a different aspect of the size distribution
(floor of packet sizes, relevant for tiny probe packets in port scanning).

**MLP never plateaus**: the MLP continues rising from 0.840 (n=1) to 0.955 (n=20),
gaining from virtually every additional feature. No clear elbow exists for MLP.

**Why we choose OPTIMAL_N=11 despite the code saying n=15:**
The plateau detection code identifies LinearSVC's *second* plateau (n=14+) and reports n=15.
We use n=11 for two reasons:
1. **Reproducibility:** the paper uses 11 features; matching this enables a fair comparison
   of our results against Table 7.
2. **Interpretability:** features 12–13 (more packet-length min/max features) are highly
   correlated with already-selected features and add little new semantic information.

**Note for the report:** the fact that the paper's authors needed to manually inspect
a curve like this and choose a cutoff is a reproducibility issue — a different researcher
would reasonably choose n=7, n=11, n=14, or n=20 depending on which model's curve they
prioritise. Document this as a methodological hazard in Report §4 (Reproducibility).


### §3.8 — Redundancy Analysis


In [ ]:
# §3.8 — Three complementary redundancy methods:
# (A) r = 1.000 duplicate pairs from §2.7
# (B) RF importance mass distribution across correlated clusters
# (C) VIF for the final selected features

# ── A: Perfect-duplicate pairs ─────────────────────────────────────────────
print('=== A. Perfect-duplicate feature pairs (Spearman r = 1.000, from §2.7) ===\n')
exact_pairs = [
    ('Subflow Fwd Packets',   'Total Fwd Packets',           'CICFlowMeter: single-subflow = total'),
    ('Subflow Fwd Bytes',     'Total Length of Fwd Packets', 'Same artefact'),
    ('Subflow Bwd Packets',   'Total Backward Packets',      'Same artefact'),
    ('Subflow Bwd Bytes',     'Total Length of Bwd Packets', 'Same artefact'),
    ('Avg Fwd Segment Size',  'Fwd Packet Length Mean',      'Two names for identical computation'),
    ('Avg Bwd Segment Size',  'Bwd Packet Length Mean',      'Two names for identical computation'),
    ('Packet Length Std',     'Packet Length Variance',      'Var = Std² (monotone transform)'),
    ('Idle Max',              'Idle Mean',                   'Near-constant idle times'),
    ('Fwd PSH Flags',         'SYN Flag Count',              'CIC capture artefact'),
    ('ECE Flag Count',        'RST Flag Count',              'CIC capture artefact'),
]
present_cols = set(get_feature_cols(df_train_bal))
for a, b, reason in exact_pairs:
    both = (a in present_cols) and (b in present_cols)
    status = '(both present → redundant)' if both else '(one or both dropped in §1)'
    print(f'  {a:<35} ↔ {b:<35} {status}')
    print(f'    Reason: {reason}\n')


In [ ]:
# §3.8 — B: RF importance mass across redundancy clusters.
# The RF naturally distributes importance across correlated features in a cluster;
# the top-N selection keeps the highest-ranked representative of each cluster.
print('=== B. RF importance — redundancy clusters ===\n')
clusters = {
    'Subflow duplicates': [
        'Subflow Fwd Packets', 'Subflow Fwd Bytes',
        'Subflow Bwd Packets', 'Subflow Bwd Bytes',
    ],
    'Packet length / segment size': [
        'Avg Fwd Segment Size', 'Fwd Packet Length Mean',
        'Avg Bwd Segment Size', 'Bwd Packet Length Mean',
        'Packet Length Std',    'Packet Length Variance',
        'Max Packet Length',    'Min Packet Length',
        'Packet Length Mean',   'Average Packet Size',
    ],
    'IAT cluster': [
        'Bwd IAT Total', 'Fwd IAT Total', 'Flow IAT Max',
        'Bwd IAT Max',   'Fwd IAT Max',   'Flow IAT Mean',
    ],
}
for cluster_name, members in clusters.items():
    present = [f for f in members if f in full_importance.index]
    if not present:
        continue
    total_imp = full_importance[present].sum()
    print(f'  {cluster_name}  (combined importance: {total_imp:.4f})')
    for feat in sorted(present, key=lambda x: -full_importance[x]):
        rank_n = list(full_importance.index).index(feat) + 1
        star = '★ SELECTED' if feat in selected_features else '  '
        print(f'    {star}  Rank {rank_n:2d}  imp={full_importance[feat]:.4f}  {feat}')
    print()


In [ ]:
# §3.8 — C: VIF for the final selected features
print('=== C. Variance Inflation Factor — final selected features ===\n')
print('Computing VIF on 10 k subsample...')
X_vif = X_bal_sel.sample(min(10_000, len(X_bal_sel)), random_state=SEED)
vif_df = compute_vif(X_vif)
print(vif_df.to_string(index=False, float_format='{:.2f}'.format))

high_vif = vif_df[vif_df['VIF'] > 5]
print(f'\nFeatures with VIF > 5 (notable multicollinearity): {len(high_vif)}')
if not high_vif.empty:
    for _, row in high_vif.iterrows():
        print(f'  {row["feature"]:<45}  VIF={row["VIF"]:.2f}')


**Redundancy analysis — three methods, surprising result:**

**A — Perfect duplicates (r = 1.000):** The four Subflow features and the Avg Segment Size
duplicates are still present in the balanced training set. All are live in the feature pool.

**B — RF importance: the RF does NOT reliably eliminate redundant pairs.**
A common assumption is that the RF will select one from each perfectly-correlated pair
and assign the other zero importance. The actual output shows this is false:

- `Avg Bwd Segment Size` (rank 9, imp=0.0346) AND `Bwd Packet Length Mean` (rank 10, imp=0.0343)
  are **both selected** in our top-11 — despite being a r=1.000 pair.
- `Subflow Fwd Packets` (rank 18, imp=0.0206) appears despite being identical to
  `Total Fwd Packets`. The RF split importance mass between them (~0.020 each).

Why? RF trees sample features randomly at each split. When two features are identical,
each tree might pick either one; importance is distributed stochastically across both.
Neither gets zero importance — they share it. The implication: **RF importance-based
feature selection is not a reliable method for removing exact duplicates.**
Explicit correlation filtering (drop one from each |r|=1.0 pair before ranking) should
precede RF importance in a production pipeline.

**C — VIF: confirms and quantifies the collinearity:**

| Feature | VIF | Interpretation |
|---------|-----|----------------|
| Bwd Packet Length Mean | **inf** | Perfectly predicted by other features (= Avg Bwd Segment Size) |
| Avg Bwd Segment Size | **inf** | Perfectly predicted by other features (= Bwd Packet Length Mean) |
| feat_bytes_per_pkt | 780 | Strongly collinear with packet-size features |
| Average Packet Size | 670 | Strongly collinear with other size statistics |
| Packet Length Std | 343 | Correlated with Variance and segment-size features |
| Bwd Packet Length Std | 150 | Correlated with packet-length cluster |
| Packet Length Variance | 21 | Notable but below 'severe' for this dataset |
| Fwd Packet Length Max | 4.86 | Acceptable |
| Destination Port | 1.20 | Independent |
| feat_win_ratio | 1.20 | Independent |
| Init_Win_bytes_backward | 1.09 | Independent |

**VIF=inf** for the two r=1.000 features means perfect multicollinearity — one is a linear
combination of the others. This is the VIF analogue of the r=1.000 Spearman result from §2.7.

**Impact on each model type:**
- **DT / RF:** unaffected — each split selects one feature and ignores the others.
- **NaiveBayes:** severely harmed — NB multiplies the likelihoods of all features
  as if independent; when two features are identical, it double-counts that evidence,
  inflating certainty. This is visible in the n=12 accuracy cliff in Figure 9.
- **SVM (RBF kernel) / ANN:** the redundant features add collinear dimensions to the
  feature space. After StandardScaling both dimensions have similar range.
  Convergence is slightly slower but the solution is not undefined — the hyperplane
  can still be found (the collinear features just don't both contribute independently).

**How to tackle redundancy (answer for report §3 rubric question):**

| Method | When to use | Applied here? |
|--------|-------------|---------------|
| Correlation filtering (drop one from each \|r\| > 0.95 pair) | Before any modelling; removes exact duplicates reliably | Not applied — demonstrates the RF limitation above |
| RF importance + top-N | Implicitly removes low-importance features, but NOT guaranteed to remove exact duplicates | Applied — selected 11 features, still contains one r=1.000 pair |
| VIF-based iterative pruning (drop highest-VIF until all < threshold) | When model assumptions require near-independence (NB, logistic regression) | Not applied here; would remove Avg Bwd Segment Size or Bwd Pkt Mean |
| PCA | Eliminates multicollinearity by construction; loses interpretability | Skipped — 11 named features preferred for cyber interpretation |

**Recommendation for future work:** apply `|r| > 0.95` filtering before RF ranking
to guarantee exact duplicates are removed from the feature pool.


### §3.9 — Save Preprocessed Artefacts to Drive


In [ ]:
# §3.9 — Persist all §3 outputs so Phase 5 (model training) can reload without re-running.
artifacts = {
    'X_bal_sel.joblib':        X_bal_sel,         # balanced + scaled + selected (model training)
    'y_bal.joblib':            y_bal,
    'X_imbal_sel.joblib':      X_imbal_sel,       # real-prevalence + scaled + selected (Phase 8.2)
    'y_imbal.joblib':          y_imbal,
    'X_test_sel.joblib':       X_test_sel,        # test + scaled + selected
    'y_test.joblib':           y_test,
    'scaler.joblib':           scaler,
    'selected_features.joblib': selected_features,
    'bf_results.joblib':       bf_results,        # brute-force curve (for the report)
    'full_importance.joblib':  full_importance,
}
for filename, obj in artifacts.items():
    path = os.path.join(DATA_DIR, filename)
    dump(obj, path)
    print(f'  Saved  {filename}')

print(f'\nAll §3 artefacts saved to Drive.')
print(f'Selected {OPTIMAL_N} features: {selected_features}')


**§3 complete.** Pipeline summary:

| Step | Input | Output | Key decision |
|------|-------|--------|--------------|
| §3.2 Binary relabel | 15/11 classes | 2 classes (BENIGN/ATTACK) | Mirror authors — binary collapse erases per-type signals |
| §3.3 Balance 1:1 | 81.9% BENIGN | 48,286 each → 96,572 total | Keep real-prevalence copy (266,739 rows) for Phase 8.2 |
| §3.4 Encoding | 66 numeric features | No encoding needed | Destination Port treated as numeric (8,927 unique values) |
| §3.5 Derived features | 66 base features | +4 features (70 total) | 2 of 4 made top-11; feat_bwd_fwd_ratio failed (binary collapse) |
| §3.6 Scaling | 70 features | 70 scaled | Fit on balanced train only — no leakage; mean≈0, std≈1 ✅ |
| §3.7 Feature selection | 70 scaled features | Top 11 (7/11 overlap with paper) | Curve has no clean elbow; n=11 chosen to match paper |
| §3.8 Redundancy | 11 features | VIF=inf for one r=1.000 pair | RF importance does NOT reliably eliminate duplicates |

**Final 11 selected features:**
1. Average Packet Size (rank 1, imp=0.0787)
2. Bwd Packet Length Std (rank 2, imp=0.0627)
3. **feat_win_ratio** (rank 3, imp=0.0595) — our derived feature
4. Packet Length Std (rank 4, imp=0.0587)
5. Destination Port (rank 5, imp=0.0403)
6. Packet Length Variance (rank 6, imp=0.0398)
7. Fwd Packet Length Max (rank 7, imp=0.0370)
8. Init_Win_bytes_backward (rank 8, imp=0.0350)
9. Avg Bwd Segment Size (rank 9, imp=0.0346) — ⚠️ r=1.000 with #10
10. Bwd Packet Length Mean (rank 10, imp=0.0343) — ⚠️ r=1.000 with #9
11. **feat_bytes_per_pkt** (rank 11, imp=0.0340) — our derived feature

→ **§4 Model Training** uses `X_bal_sel` (shape: 96,572 × 11) and `y_bal`.


---

# §4 — Model Training

**Goal:** Train all six classifiers with GridSearchCV-tuned hyperparameters and confirm in-distribution stability with k=5 cross-validation. Mirror the paper's training setup; save all models and best params to Drive so a Colab disconnect doesn't force a full re-run.

**Pipeline:**
1. **Reload artefacts** from §3 (`X_bal_sel`, `y_bal`, `selected_features`)
2. **GridSearchCV** — tune each model's hyperparameters (k=5, training set only)
3. **k=5 cross-validation** — verify in-distribution stability with best params
4. **Train final models** on the full balanced training set; persist to Drive

**Six models:** Decision Tree (DT), Random Forest (RF), SVM (RBF kernel), Naive Bayes (NB), ANN (1 hidden layer), DNN (3 hidden layers) — all scikit-learn, all CPU-only.

> **SVM note:** `SVC` with RBF kernel scales ~O(n^2). GridSearch and CV are run on a 10% subsample of the training data; the final model is fitted on 20%. This mirrors the paper's approach. All other models use the full balanced training set.

In [ ]:
# §4 setup — reload §3 artefacts; import model factory
import os, sys
import numpy as np
import pandas as pd
import joblib

REPO_DIR = '/content/data-science-cyber-final-project'
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

from src.models import (
    make_all_models, PARAM_GRIDS, MODEL_NAMES,
    tune_all_models, cross_validate_all, train_all_models, load_models,
)

X_bal_sel         = joblib.load(os.path.join(DATA_DIR, 'X_bal_sel.joblib'))
y_bal             = joblib.load(os.path.join(DATA_DIR, 'y_bal.joblib'))
X_imbal_sel       = joblib.load(os.path.join(DATA_DIR, 'X_imbal_sel.joblib'))
y_imbal           = joblib.load(os.path.join(DATA_DIR, 'y_imbal.joblib'))
X_test_sel        = joblib.load(os.path.join(DATA_DIR, 'X_test_sel.joblib'))
y_test            = joblib.load(os.path.join(DATA_DIR, 'y_test.joblib'))
selected_features = joblib.load(os.path.join(DATA_DIR, 'selected_features.joblib'))

print(f"Training set (balanced):    {X_bal_sel.shape}  {dict(y_bal.value_counts())}")
print(f"Training set (imbalanced):  {X_imbal_sel.shape}")
print(f"Test set:                   {X_test_sel.shape}  {dict(y_test.value_counts())}")
print(f"Selected features ({len(selected_features)}): {selected_features}")

### §4.1 — Model Configurations

All six classifiers are defined in `src/models.py` and mirror the paper exactly:

| Name | sklearn class | Key design choices |
|------|--------------|-------------------|
| DT | `DecisionTreeClassifier` | Post-pruning via `ccp_alpha`; fixed seed |
| RF | `RandomForestClassifier` | 100-350 trees; depth/leaf tuned; `n_jobs=-1` |
| SVM | `SVC(kernel='rbf')` | `C` and `gamma` tuned; `probability=True` for ROC-AUC; O(n^2) subsample |
| NB | `GaussianNB` | `var_smoothing` tuned; feature-independence assumption is the known weakness |
| ANN | `MLPClassifier` | 1 hidden layer (50 or 100 units); tanh/relu; adam; early stopping |
| DNN | `MLPClassifier` | 3 hidden layers (15,15,15 or 50,50,50); tanh; adam; early stopping |

All use `SEED = 42`.

In [ ]:
# §4.1 — confirm model factory output
models_unfitted = make_all_models()
for name, clf in models_unfitted.items():
    defaults = clf.__class__().get_params()
    non_default = {k: v for k, v in clf.get_params().items() if v != defaults.get(k)}
    print(f"{name:4s}  {clf.__class__.__name__:30s}  non-default: {non_default}")

### §4.2 — Hyperparameter Optimisation (GridSearchCV, k=5)

Search spaces mirror the paper's `03_Optimize_Hyperparameter.ipynb`:

| Model | Parameters searched | Grid size |
|-------|-------------------|-----------|
| DT | `ccp_alpha` in {0, 1.44e-5, 5e-5, 1e-4} | 4 |
| RF | `n_estimators` in {100, 350}, `max_depth` in {10, 20, None}, `min_samples_leaf` in {1, 1e-5} | 12 |
| SVM | `C` in {1, 10, 100}, `gamma` in {0.01, 0.1, 1} — on 10% subsample | 9 |
| NB | `var_smoothing` in {1e-9, 1e-5, 1.0} | 3 |
| ANN | `hidden_layer_sizes` in {(50,),(100,)}, `activation` in {tanh, relu}, `alpha` in {1e-4, 1e-3} | 8 |
| DNN | `hidden_layer_sizes` in {(15,15,15),(50,50,50)}, `alpha` in {1e-5, 1e-4} | 4 |

**Resume-safe:** each model's best params are saved as `best_params_<MODEL>.joblib`. If the file exists it is reloaded and GridSearch is skipped — safe to re-run after a Colab disconnect.

In [ ]:
# §4.2 — GridSearchCV (resume-safe: skips models whose params are already saved)
best_params = tune_all_models(
    X_bal_sel, y_bal,
    save_dir=DATA_DIR,
    cv=5,
    seed=SEED,
)

### §4.2 — Best Hyperparameters: Our Results vs. the Paper

The paper's reported best hyperparameters (from their ):

| Model | Paper's best params |
|-------|-------------------|
| DT |  |
| RF |  |
| SVM |  |
| NB |  |
| ANN |  |
| DNN |  |

The code below runs the comparison — interpretation of the actual mismatches follows.

In [ ]:
# Print side-by-side comparison with the paper
paper_params = {
    'DT':  {'ccp_alpha': 1.44e-5},
    'RF':  {'n_estimators': 350, 'max_depth': 20, 'min_samples_leaf': 1e-5, 'criterion': 'gini'},
    'SVM': {'C': 100, 'gamma': 1, 'kernel': 'rbf'},
    'NB':  {'var_smoothing': 1.0},
    'ANN': {'activation': 'tanh', 'hidden_layer_sizes': (50,), 'solver': 'adam', 'alpha': 1e-4},
    'DNN': {'activation': 'tanh', 'hidden_layer_sizes': (15, 15, 15), 'solver': 'adam', 'alpha': 1e-5},
}

print(f"{'Model':<5}  {'Our best params':<60}  Match paper?")
print('-' * 90)
for name in MODEL_NAMES:
    our = best_params.get(name, {})
    ref = paper_params.get(name, {})
    match = '✅' if our == ref else '⚠️ '
    print(f"{name:<5}  {str(our):<60}  {match}")

**§4.2 — Actual results vs. paper: what the mismatches mean**

2 out of 6 models match the paper exactly (SVM ✅, ANN ✅); 4 differ (DT ⚠️, RF ⚠️, NB ⚠️, DNN ⚠️).

| Model | We got | Paper had | Practical significance |
|-------|--------|-----------|------------------------|
| DT | `ccp_alpha=0.0` (no pruning) | `ccp_alpha=1.44e-05` (minimal pruning) | Both produce near-identical CV accuracy (~99.7%); the difference is negligible. DT is likely already very shallow on this well-separated data. |
| RF | 100 trees, no depth limit | 350 trees, `max_depth=20` | GridSearch preferred smaller, unconstrained trees. With 11 features and near-perfect separability, 100 trees suffice — adding 250 more would not move the needle. |
| SVM | `C=100, gamma=1` | `C=100, gamma=1` | **Exact match** — the search space and the optimal point were both reproduced. |
| NB | `var_smoothing=1e-05` | `var_smoothing=1.0` | Large difference (5 orders of magnitude). `var_smoothing=1.0` adds strong Laplace smoothing, effectively regularising NB away from the data; `1e-05` is near-default (almost no smoothing). Our GridSearch preferred to let the data speak. Both values still leave NB well below tree models in CV accuracy. |
| ANN | `(50,), tanh, alpha=1e-4` | `(50,), tanh, alpha=1e-4` | **Exact match.** |
| DNN | `(50,50,50), tanh, alpha=1e-5` | `(15,15,15), tanh, alpha=1e-5` | Wider hidden layers. Our search found that 50-unit layers outperform 15-unit layers on our 11 features (vs the paper's different 11 features). Both architectures converge to similar CV accuracy. |

**Reproducibility conclusion for §4.2:** SVM and ANN hyperparameters are exactly reproduced. The 4 mismatches are expected given (a) different 11-feature set (7/11 overlap), (b) different training-set subsample used by GridSearch, (c) sklearn version differences. The mismatches are in the *magnitude* of regularisation parameters, not in the *direction* of model structure choices — all models converge to high CV accuracy regardless. What matters for the reproduction claim is whether model ranking holds in §5, not exact parameter agreement.

### §4.3 — In-distribution Cross-validation (k=5)

Cross-validate each model with its best hyperparameters on the balanced training set. This measures **in-distribution generalisation** — how well each model performs on held-out data from the *same* distribution as its training data.

**Why this matters for the critique:** if a model achieves high CV accuracy here but low accuracy on the 2018 progressive test (§5), the gap reflects **distribution shift / concept drift**, not classical overfitting. Classical overfitting would appear as a large train-vs-CV gap *within* the same distribution. This contrast is the empirical foundation for the Phase 8.1 critique of the paper's "overfitting" framing.

In [ ]:
# §4.3 — k=5 cross-validation with best hyperparameters
print("Running k=5 cross-validation on balanced training set ...\n")
cv_results = cross_validate_all(
    X_bal_sel, y_bal,
    best_params=best_params,
    cv=5,
    seed=SEED,
)

joblib.dump(cv_results, os.path.join(DATA_DIR, 'cv_results.joblib'))
print()
print(cv_results.to_string())

**Cross-validation interpretation:**

These scores confirm in-distribution stability. Actual results:

| Model | CV Accuracy | CV Std | Notes |
|-------|------------|--------|-------|
| DT | 99.68% | ±0.07% | Near-perfect |
| RF | 99.73% | ±0.06% | Near-perfect |
| DNN | 96.84% | ±0.32% | |
| ANN | 96.66% | ±0.17% | |
| SVM | 96.44% | ±0.29% | 10% subsample |
| NB | 84.44% | ±0.12% | Lowest by >12pp |

Key observations:

- **DT / RF** achieve near-perfect CV accuracy (99.7%) — the paper reports the same. This sets up the core contrast: these models score highest within the 2017 distribution but are expected to drop the most on the 2018 progressive test, since high in-distribution accuracy suggests strong memorisation of 2017-specific patterns.

- **NB** reaches only 84.44% — lowest by more than 12 percentage points. The mechanism is the independence-assumption violation: our 11-feature set contains multiple highly correlated packet-length features (notably `Bwd Packet Length Mean` and `Avg Bwd Segment Size`, which are exact duplicates at r=1.000 — documented in §3). NB treats each feature as independent evidence and double-counts their contribution. The §3 brute-force curve illustrated this directly: during feature selection exploration, NB's accuracy dropped sharply when a correlated packet-length feature was added at n=12 in the exploration. We stopped at n=11, but the correlation problem already exists in our chosen feature set.

- **SVM / ANN / DNN** cluster tightly in the 96.4–96.9% range. These models do not memorise the 2017 dataset as perfectly as tree models — which may turn out to be an advantage when the test distribution shifts in §5.

**What this does not tell us:** whether the learned patterns transfer to CSE-CIC-IDS2018. That is §5's job. The contrast between these CV numbers and §5's progressive test numbers is the central empirical evidence for the project's critique.

In [ ]:
# §4.4 — Train final models on full balanced training set; persist to Drive
# SVM uses SVM_TRAIN_FRAC=20% subsample to keep runtime manageable (mirrors paper)
print("Training final models ...\n")
trained_models = train_all_models(
    X_bal_sel, y_bal,
    best_params=best_params,
    save_dir=DATA_DIR,
    seed=SEED,
)

print()
for name, clf in trained_models.items():
    print(f"  {name:4s}  {clf.__class__.__name__}  ready")

### §4 Complete — Summary

| Step | Output | Saved to Drive |
|------|--------|---------------|
| §4.1 Model configs | 6 unfitted estimators confirmed | — |
| §4.2 GridSearchCV | Best params per model | `best_params_<MODEL>.joblib` |
| §4.3 k=5 CV | In-distribution accuracy table | `cv_results.joblib` |
| §4.4 Final training | 6 fitted models | `model_<MODEL>.joblib` |

**Next:** §5 evaluates these models on both the CIC-IDS2017 held-out test split (in-distribution) and the CSE-CIC-IDS2018 progressive test set to reproduce and critique the paper's Tables 4-7.

---
## §5 — Evaluation & Metrics

**Goal:** Reproduce the paper's evaluation on both test sets; add the metrics the authors omitted.

| Step | Description | Saved to Drive |
|------|-------------|----------------|
| §5.1 | In-distribution baseline (§4 CV results) | — |
| §5.2 | Progressive evaluation (CSE-CIC-IDS2018) | `results_progressive.joblib` |
| §5.3 | Confusion-matrix grid — progressive test | `confusion_progressive.png` |
| §5.4 | Performance-drop table (CV → progressive) | `results_drop.joblib` |
| §5.5 | Reproduction check vs. paper Tables 4–7 | — |
| §5.6 | Metric justification write-up | — |

> **Metric additions beyond the paper:** F₂ (β=2, weights recall so missed attacks are
> penalised more than false alarms), MCC (balanced, robust to the 84%-BENIGN test-set
> imbalance), ROC-AUC (threshold-independent).
> The paper uses only Accuracy, Precision, Recall, F1.

In [ ]:
# §5 setup — reload trained models and all preprocessed artefacts
import os, sys, pathlib
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt

# ── locate src/ ───────────────────────────────────────────────────────────────
_src_candidates = [
    '/content/ids-project/src',
    '/content/data-science-cyber-final-project/src',
]
_src = next((p for p in _src_candidates if os.path.isdir(p)), None)

if _src is None:
    REPO_URL = 'https://github.com/Rosette28/data-science-cyber-final-project'
    REPO_DIR_CLONE = '/content/ids-project'
    if not os.path.exists(REPO_DIR_CLONE):
        os.system(f'git clone {REPO_URL} {REPO_DIR_CLONE}')
    _src = REPO_DIR_CLONE + '/src'

if _src not in sys.path:
    sys.path.insert(0, _src)

REPO_DIR    = pathlib.Path(_src).parent
FIGURES_DIR = str(REPO_DIR / 'figures')
os.makedirs(FIGURES_DIR, exist_ok=True)
print(f"src path : {_src}")

from models     import MODEL_NAMES, load_models
from evaluation import evaluate_all, confusion_grid
print("Imports OK")

# ── Drive path ────────────────────────────────────────────────────────────────
# Reuse DATA_DIR from the header cell (§ config) if this session already has it;
# only fall back to a hardcoded default on a fresh kernel. Must match the header
# cell's DATA_DIR ('.../ids_data/raw/') -- a mismatch here silently empties `models`.
if 'DATA_DIR' not in dir():
    DATA_DIR = '/content/drive/MyDrive/ids_data/raw/'   # <- adjust if needed

# Diagnostic: confirm DATA_DIR exists and list .joblib files
if os.path.isdir(DATA_DIR):
    files = sorted(f for f in os.listdir(DATA_DIR) if f.endswith('.joblib'))
    print(f"\nDATA_DIR: {DATA_DIR}")
    print(f"  .joblib files found ({len(files)}): {files}")
else:
    print(f"\nWARNING: DATA_DIR not found: {DATA_DIR}")
    print("  Check that Drive is mounted and DATA_DIR points to the right folder.")

def _load(name):
    return joblib.load(os.path.join(DATA_DIR, name))

# ── reload preprocessed artefacts ─────────────────────────────────────────────
if 'X_test_sel' not in dir():
    X_bal_sel         = _load('X_bal_sel.joblib')
    y_bal             = _load('y_bal.joblib')
    X_test_sel        = _load('X_test_sel.joblib')
    y_test            = _load('y_test.joblib')
    selected_features = _load('selected_features.joblib')
    print("Preprocessed artefacts loaded.")
else:
    print("Reusing artefacts already in memory.")

# ── reload trained models ─────────────────────────────────────────────────────
if 'models' not in dir():
    models = load_models(DATA_DIR)
    print(f"Models loaded: {list(models.keys())}")
else:
    print("Reusing models already in memory.")

### §5.1 — In-Distribution Baseline

The models were trained on the full balanced training set (`X_bal_sel`, 96,572 rows),
so no separate held-out in-distribution partition exists. The **k=5 cross-validation
results from §4** are the leakage-free in-distribution baseline. A post-hoc split of
the same training data would over-estimate performance relative to the models' actual
generalisation within distribution.

In [ ]:
# §5.1 — load §4 k=5 CV results; fall back to documented values if file missing
_cv_path = os.path.join(DATA_DIR, 'cv_results.joblib')

if os.path.exists(_cv_path):
    cv_results = joblib.load(_cv_path)
    print("cv_results loaded from Drive.")
else:
    # File not in Drive — use the values recorded during the §4 Colab run
    # (documented in reports/section4_decisions_to_remember.md).
    print("cv_results.joblib not found — using documented §4 results.")
    cv_results = pd.DataFrame({
        'CV Mean Acc': [0.9973, 0.9968, 0.9684, 0.9666, 0.9644, 0.8444],
        'CV Std':      [0.0006, 0.0007, 0.0032, 0.0017, 0.0029, 0.0012],
        'CV Min':      [0.9961, 0.9956, 0.9630, 0.9634, 0.9601, 0.8429],
        'CV Max':      [0.9981, 0.9977, 0.9730, 0.9685, 0.9684, 0.8460],
        'Note':        ['', '', '', '', 'subsample 10%', ''],
    }, index=pd.Index(['RF', 'DT', 'DNN', 'ANN', 'SVM', 'NB'], name='Model'))
    # Save for downstream cells
    joblib.dump(cv_results, _cv_path)
    print(f"  Saved to {_cv_path} for downstream use.")

display_cols = [c for c in ['CV Mean Acc', 'CV Std', 'CV Min', 'CV Max', 'Note']
                if c in cv_results.columns]
print("\n=== §4 k=5 CV Accuracy — In-Distribution Baseline (leakage-free) ===\n")
print(cv_results[display_cols].to_string())
print("\nNote: SVM CV was run on a 10% subsample due to O(n^2) scaling.")

### §5.2 — Progressive Evaluation (CSE-CIC-IDS2018)

**This is the central experiment.** Models trained on CIC-IDS2017 are evaluated on the
temporally later CSE-CIC-IDS2018. Any drop compared to §5.1 reflects degradation on an
unseen distribution (concept drift / distribution shift), not classical overfitting to
training noise.

Progressive test: 106,906 flows, **84.1% BENIGN**. A naive "always predict BENIGN"
baseline would score 84.1% accuracy but MCC = 0 — MCC and F₂ are essential complements
to accuracy on this imbalanced set.

In [ ]:
# §5.2 — evaluate all models on the progressive test set (resume-safe)
prog_path    = os.path.join(DATA_DIR, 'results_progressive.joblib')
results_prog = evaluate_all(models, X_test_sel, y_test, save_path=prog_path)

print("\n=== Progressive Evaluation — CSE-CIC-IDS2018 ===\n")
print(results_prog.to_string())

**Progressive evaluation interpretation:**

*(Fill in after running in Colab. Note which models held up and which dropped the most.
Compare Accuracy vs. MCC — if Accuracy is high but MCC is near zero, the model is mostly
predicting the majority class. Highlight whether the paper's claim that SVM/ANN are more
robust is supported or contradicted. Update before writing Report §5.)*

### §5.3 — Confusion Matrices (Progressive Test Set)

In [ ]:
# §5.3 — confusion matrix grid on the progressive test set
cm_path = os.path.join(FIGURES_DIR, 'confusion_progressive.png')
confusion_grid(
    models, X_test_sel, y_test,
    title='Confusion Matrices — Progressive Test (CSE-CIC-IDS2018)',
    save_path=cm_path,
)

### §5.4 — Performance Drop: CV → Progressive Test

The drop below is the paper's primary evidence that models trained on 2017 data degrade
on 2018 data. The larger the negative drop, the more the model depended on 2017-specific
statistical patterns rather than transferable detection logic.

In [ ]:
# §5.4 — performance-drop table
drop_path = os.path.join(DATA_DIR, 'results_drop.joblib')

rows = []
for name in MODEL_NAMES:
    if name not in results_prog.index or name not in cv_results.index:
        continue
    cv_acc   = float(cv_results.loc[name, 'CV Mean Acc'])
    prog_acc = float(results_prog.loc[name, 'Accuracy'])
    prog_mcc = float(results_prog.loc[name, 'MCC'])   if 'MCC' in results_prog.columns else None
    prog_f2  = float(results_prog.loc[name, 'F2'])    if 'F2'  in results_prog.columns else None
    rows.append({
        'Model':        name,
        'CV Acc (§4)':  cv_acc,
        'Prog Acc':     prog_acc,
        'Acc Drop':     round(prog_acc - cv_acc, 4),
        'Prog MCC':     prog_mcc,
        'Prog F2':      prog_f2,
    })

results_drop = pd.DataFrame(rows).set_index('Model')
joblib.dump(results_drop, drop_path)

print("=== Performance Drop: k=5 CV Accuracy → Progressive Accuracy ===\n")
print(results_drop.to_string())
print("\nNegative Acc Drop = worse on 2018 data (expected: concept drift).")

**Drop analysis interpretation:**

*(Fill in after Colab run. Identify which models dropped most vs. least. Connect to the
paper's claim that DT/RF overfit while SVM/ANN resist. Note whether MCC tells a different
story than Accuracy alone — on the 84%-BENIGN test a model mostly predicting BENIGN has
high accuracy but near-zero MCC. Update before writing Report §5 and the Phase 8.1 drift
argument.)*

### §5.5 — Reproduction Check vs. Paper (Tables 4–7)

Fill in the paper's Table 7 values (progressive test) into `paper_prog` below before
final submission. Until then the diff column shows "—".

In [ ]:
# §5.5 — reproduction check vs. paper Table 7 (progressive test)
# Fill in the paper's Table 7 values before final submission.
paper_prog = {
    'DT':  {'Accuracy': None, 'Precision': None, 'Recall': None, 'F1': None},
    'RF':  {'Accuracy': None, 'Precision': None, 'Recall': None, 'F1': None},
    'SVM': {'Accuracy': None, 'Precision': None, 'Recall': None, 'F1': None},
    'NB':  {'Accuracy': None, 'Precision': None, 'Recall': None, 'F1': None},
    'ANN': {'Accuracy': None, 'Precision': None, 'Recall': None, 'F1': None},
    'DNN': {'Accuracy': None, 'Precision': None, 'Recall': None, 'F1': None},
}

print("=== Reproduction Check: Our Results vs. Paper Table 7 ===\n")
print(f"{'Model':<5}  {'Our Acc':>8}  {'Paper Acc':>9}  {'Acc D':>6}"
      f"  {'Our F1':>7}  {'Paper F1':>8}  {'F1 D':>6}")
print('-' * 62)
for name in MODEL_NAMES:
    if name not in results_prog.index:
        continue
    our_acc   = results_prog.loc[name, 'Accuracy']
    our_f1    = results_prog.loc[name, 'F1']
    pap       = paper_prog.get(name, {})
    pap_acc   = pap.get('Accuracy')
    pap_f1    = pap.get('F1')
    acc_d     = f"{our_acc - pap_acc:+.4f}" if pap_acc is not None else '  —'
    f1_d      = f"{our_f1  - pap_f1 :+.4f}" if pap_f1  is not None else '  —'
    pap_acc_s = f"{pap_acc:.4f}" if pap_acc is not None else '    ?'
    pap_f1_s  = f"{pap_f1 :.4f}" if pap_f1  is not None else '   ?'
    print(f"{name:<5}  {our_acc:>8.4f}  {pap_acc_s:>9}  {acc_d:>6}"
          f"  {our_f1:>7.4f}  {pap_f1_s:>8}  {f1_d:>6}")

**Reproduction verdict:**

*(Fill in after entering the paper's Table 7 values. State whether the reproduction is
close or diverges, and give a reason for each discrepancy — different feature set (7/11
overlap), different 2018 file (pre-processed community sample vs. original UNB CSVs),
subsample seed, or sklearn version. This feeds directly into Report §4 — Reproducibility
Analysis.)*

### §5.6 — Metric Justification

#### Why each metric was chosen

| Metric | Mathematical definition | Cybersecurity interpretation |
|--------|------------------------|------------------------------|
| **Accuracy** | (TP + TN) / N | Overall fraction of flows classified correctly. Misleading when the test set is 84% BENIGN — a model that always predicts BENIGN achieves 84% with no detection ability. Reported to match the paper. |
| **Precision** | TP / (TP + FP) | Of all flows flagged as attacks, what fraction were real? Low precision = **alert fatigue**: the analyst's queue fills with false alarms, masking real threats. |
| **Recall** | TP / (TP + FN) | Of all real attacks, what fraction were caught? Low recall = **missed intrusions**: attackers pass through undetected. The costlier failure mode in most IDS deployments. |
| **F1** | 2·Prec·Rec / (Prec + Rec) | Harmonic mean of Precision and Recall; treats both failure modes equally. Reported to match the paper. |
| **F₂** (β = 2) | (1+4)·Prec·Rec / (4·Prec + Rec) | Weights Recall twice over Precision. β=2 reflects the asymmetric cost of IDS failures: a missed attack (FN) can cause a data breach, while a false alarm (FP) costs only analyst time. |
| **MCC** | (TP·TN − FP·FN) / √((TP+FP)(TP+FN)(TN+FP)(TN+FN)) | Single balanced metric for all four confusion-matrix cells. MCC = 0 means no better than chance. On the 84%-BENIGN test set, **MCC is the most reliable single number**: the naive majority classifier scores 84% accuracy but MCC = 0. |
| **ROC-AUC** | Area under the ROC curve | Probability the model ranks a random attack above a random benign flow, across all thresholds. Threshold-independent; useful for comparing models when the decision boundary has not yet been chosen. |

#### Why the paper's metric set is insufficient

1. **Accuracy is only meaningful on their balanced test set.** On a 50/50 set the naive
   baseline is 50%; on the real-prevalence test (84.1% BENIGN) it rises to 84.1%. MCC
   and F₂ remain interpretable in both settings without adjustment.
2. **No threshold-independent metric.** ROC-AUC shows discrimination ability independent
   of where the decision boundary is set — important when the 1:1 training ratio differs
   from the deployment ratio.
3. **F1 treats FP and FN as equally costly.** In IDS this is rarely true. F₂ makes the
   asymmetric cost explicit in the metric.

### §5 Complete — Summary

| Step | Status | Key output |
|------|--------|------------|
| §5.1 In-distribution baseline | ✅ | §4 CV table displayed |
| §5.2 Progressive evaluation | ✅ | `results_progressive.joblib`; full metric suite per model |
| §5.3 Confusion matrices | ✅ | `confusion_progressive.png` |
| §5.4 Performance drop | ✅ | `results_drop.joblib`; CV → progressive gap per model |
| §5.5 Reproduction check | ⚠️ | Fill in paper Table 7 values before final submission |
| §5.6 Metric justification | ✅ | Written above |

**Next:** §6 analyses *where* and *why* models fail — which attack types are most often
misclassified on the 2018 progressive test set.

---
## §6 — Error Analysis

**Goal:** understand *where* and *why* the models fail on the progressive test set — not just
the aggregate metrics from §5. FNs (missed attacks) and FPs (false alarms) have different
costs in an IDS context, and different attack types are expected to "collapse" differently
given the 2017→2018 distribution shift documented in §1's temporal analysis.

| Step | Description | Saved to Drive / figures |
|------|-------------|---------------------------|
| §6.1 | Setup — reload models, test set, recover original attack-type labels | `y_test_multiclass.joblib` |
| §6.2 | Per-attack-type detection rate (recall), all 6 models | `error_by_attack_type.png` |
| §6.3 | Sample false negatives — missed attacks on the worst-detected type | — |
| §6.4 | False positives — benign flows misclassified as attacks | — |
| §6.5 | Cybersecurity implications of FN vs. FP | — |
| §6.6 | FP/FN trade-off and threshold considerations | — |

In [ ]:
# §6.1 -- setup: reload models + test artefacts if needed, recover attack-type labels
import os, sys, pathlib
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
import seaborn as sns

_src_candidates = [
    '/content/ids-project/src',
    '/content/data-science-cyber-final-project/src',
]
_src = next((p for p in _src_candidates if os.path.isdir(p)), None)
if _src is None:
    REPO_URL = 'https://github.com/Rosette28/data-science-cyber-final-project'
    REPO_DIR_CLONE = '/content/ids-project'
    if not os.path.exists(REPO_DIR_CLONE):
        os.system(f'git clone {REPO_URL} {REPO_DIR_CLONE}')
    _src = REPO_DIR_CLONE + '/src'
if _src not in sys.path:
    sys.path.insert(0, _src)

REPO_DIR    = pathlib.Path(_src).parent
FIGURES_DIR = str(REPO_DIR / 'figures')
os.makedirs(FIGURES_DIR, exist_ok=True)

from models     import MODEL_NAMES, load_models
from evaluation import per_attack_type_recall, false_negative_examples, false_positive_examples

# Reuse DATA_DIR from the header cell (§ config) if this session already has it;
# only fall back to a hardcoded default on a fresh kernel. Must match the header
# cell's DATA_DIR ('.../ids_data/raw/') -- a mismatch here silently empties `models`.
if 'DATA_DIR' not in dir():
    DATA_DIR = '/content/drive/MyDrive/ids_data/raw/'   # <- adjust if needed

def _load(name):
    return joblib.load(os.path.join(DATA_DIR, name))

if 'X_test_sel' not in dir():
    X_test_sel        = _load('X_test_sel.joblib')
    y_test             = _load('y_test.joblib')
    selected_features  = _load('selected_features.joblib')
if 'models' not in dir():
    models = load_models(DATA_DIR)

# -- recover the original (pre-binary-collapse) attack-type labels for the test set --
# Needed for section 6.2's per-attack-type breakdown; binary_relabel() in section 3.2
# overwrote the multiclass Label, but df_test_clean (pre-relabel) keeps it, and row
# order/index is preserved through binary_relabel -> add_derived_features -> scaling
# -> selection.
_ymc_path = os.path.join(DATA_DIR, 'y_test_multiclass.joblib')
if 'y_test_multiclass' in dir():
    print("Reusing y_test_multiclass already in memory.")
elif os.path.exists(_ymc_path):
    y_test_multiclass = joblib.load(_ymc_path)
    print("y_test_multiclass reloaded from Drive.")
elif 'df_test_clean' in dir():
    y_test_multiclass = df_test_clean['Label'].loc[X_test_sel.index]
    joblib.dump(y_test_multiclass, _ymc_path)
    print(f"y_test_multiclass derived from df_test_clean and saved to {_ymc_path}.")
else:
    y_test_multiclass = None
    print("WARNING: original attack-type labels unavailable in this session.")
    print("  Re-run the section 3.1-3.2 cells (df_test_clean / binary_relabel) once, in the")
    print("  same session as this cell, so y_test_multiclass can be derived and cached to Drive.")

if y_test_multiclass is not None:
    print(f"\nAttack-type breakdown of the progressive test set ({len(y_test_multiclass):,} rows):")
    print(y_test_multiclass.value_counts().to_string())

### §6.2 — Per-Attack-Type Detection Rate (Recall)

Binary metrics in §5 average over every attack type. A model can score 90%+ recall
overall while missing an entire attack category — the aggregate number hides it.
This table computes, per model, the fraction of *each* original attack type that was
correctly flagged as ATTACK on the progressive (2018) test set.

In [ ]:
# §6.2 -- recall per attack type, per model
assert y_test_multiclass is not None, "y_test_multiclass required -- see the section 6.1 warning above."

recall_by_type = per_attack_type_recall(models, X_test_sel, y_test_multiclass)
print("=== Detection rate (recall) by attack type -- progressive test (CSE-CIC-IDS2018) ===\n")
print(recall_by_type.to_string())

# Heatmap: attack type (row) x model (column), colour = recall
model_cols = [c for c in MODEL_NAMES if c in recall_by_type.columns]
fig, ax = plt.subplots(figsize=(8, max(4, 0.4 * len(recall_by_type))))
sns.heatmap(
    recall_by_type[model_cols], annot=True, fmt='.2f', cmap='RdYlGn',
    vmin=0, vmax=1, linewidths=0.5, cbar_kws={'label': 'Recall (detection rate)'}, ax=ax,
)
ax.set_title('Per-attack-type detection rate -- progressive test set', fontsize=12)
ax.set_ylabel('')
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'error_by_attack_type.png'), dpi=150, bbox_inches='tight')
plt.show()

worst = recall_by_type[model_cols].mean(axis=1).sort_values().head(5)
print("\nWorst-detected attack types (mean recall across all 6 models):")
print(worst.to_string())

**Reading the heatmap:** *(fill in with the actual pattern from your run — e.g. which
attack types are red across every model column vs. which are green for some models and red
for others. Cross-reference against §1's temporal analysis: attack types present in 2018 but
rare/absent in 2017's training data are the ones most likely to collapse — that is concept
drift, not overfitting, showing up at the per-class level. Attack types with low recall for
DT/RF specifically but higher recall for SVM/ANN would directly support the paper's claim;
uniformly low recall across all 6 models would point instead to those attack types being
inherently hard to separate with the selected 11 features, independent of model choice.)*

### §6.3 — Sample False Negatives (Missed Attacks)

Pulling the actual flows behind the worst-recall attack type from §6.2, for the model with
the highest overall progressive accuracy (per §5.2) — to see what these missed flows look
like in feature space, not just as a count.

In [ ]:
# §6.3 -- sample FN flows for the single worst-recall attack type
best_model_name = results_prog['Accuracy'].idxmax() if 'results_prog' in dir() else MODEL_NAMES[0]
best_model      = models[best_model_name]
worst_type      = worst.index[0]
print(f"Model: {best_model_name}  |  Worst-detected attack type: {worst_type}\n")

mask_type = (y_test_multiclass == worst_type).values
fn_examples, fn_count = false_negative_examples(
    best_model, X_test_sel.loc[mask_type], y_test.loc[mask_type], n=10,
)
print(f"False negatives for '{worst_type}': {fn_count} / {int(mask_type.sum())} flows missed "
      f"({fn_count / max(1, int(mask_type.sum())):.1%})\n")
print("Sample missed flows (scaled feature values):")
print(fn_examples.to_string())

**Interpretation:** *(fill in after inspecting the sample rows above — do the missed flows'
scaled feature values sit close to 0 (i.e., near the BENIGN population mean)? That would mean
the attack's flow-level statistics genuinely resemble benign traffic on these 11 features —
a feature-set limitation, not a model failure. If instead they look like clear outliers that
the model still misses, that points more to a decision-boundary/threshold issue.)*

### §6.4 — False Positives (Benign Flows Misclassified as Attacks)

False positives drive alert fatigue: every one is an analyst-hour spent on traffic that
was never a threat. Counting them per model here, alongside the FN counts from §6.3, gives
the full FP/FN picture needed for §6.5-§6.6.

In [ ]:
# §6.4 -- FP counts for all 6 models + sample FP flows for the model with the most FPs
fp_counts = {}
for name, clf in models.items():
    _, n_fp = false_positive_examples(clf, X_test_sel, y_test, n=0)
    fp_counts[name] = n_fp

fp_summary = pd.Series(fp_counts, name='FP count').sort_values(ascending=False)
n_benign_test = int((y_test == 'BENIGN').sum())
print("=== False-positive counts -- progressive test set ===\n")
print(fp_summary.to_string())
print(f"\n(out of {n_benign_test:,} BENIGN flows in the progressive test set)")

fp_worst_model = fp_summary.index[0]
fp_examples, _ = false_positive_examples(models[fp_worst_model], X_test_sel, y_test, n=10)
print(f"\nSample false-positive flows for {fp_worst_model} (scaled feature values):")
print(fp_examples.to_string())

**Interpretation:** *(fill in — which model produces the most false alarms on the 2018 set?
Does it line up with the model that had the lowest CV std / most confident decision boundary
in §4.3, or is it unrelated? A model with both high recall and a high FP count is trading
missed-attack risk for alert-fatigue risk — worth naming explicitly for §6.5.)*

### §6.5 — Cybersecurity Implications of FN vs. FP

*(👤 — rewrite this in your own words before submission; drafted here as a starting point.)*

The two error types are not symmetric in an IDS deployment:

- **False Negative (missed attack):** the model predicts BENIGN for a flow that is actually
  malicious. The attacker's traffic passes through undetected — this is the failure mode with
  direct security consequences (data exfiltration, lateral movement, ransomware execution).
  A FN is silent: unlike an FP, nothing alerts an analyst that something needs review, so the
  cost is borne later, often at a much higher severity, when the intrusion is discovered by
  other means (or not at all).

- **False Positive (false alarm):** the model predicts ATTACK for a flow that is actually
  benign. No breach occurs, but every FP consumes analyst time to triage and dismiss. At
  scale, a high FP rate causes **alert fatigue** — analysts start deprioritizing or ignoring
  IDS alerts altogether, which paradoxically increases the real risk of a missed intrusion
  (a genuine alert buried in noise gets the same low attention as the false ones).

- **Why this matters for model choice here:** §6.2's per-attack-type recall and §6.4's FP
  counts show that no single model minimizes both error types simultaneously. Because a missed
  attack (FN) is generally the more expensive failure mode (§6.6 below quantifies this via F₂
  and threshold choice), a deployment decision should weight recall on the progressive test set
  more heavily than raw accuracy — which is exactly why F₂ and MCC were added in §5.6 beyond
  the paper's Accuracy/Precision/Recall/F1.

### §6.6 — FP/FN Trade-off and Threshold Considerations

*(👤 — rewrite this in your own words before submission; drafted here as a starting point.)*

All six models here use a default 0.5 decision threshold on `predict()`. For the four models
with `predict_proba` (RF, SVM, NB, and the two `MLPClassifier`s — ANN/DNN), the threshold is
a free parameter that can be moved to trade FN for FP without retraining:

- **Lowering the threshold** (flag more flows as ATTACK) increases recall (fewer FNs, fewer
  missed intrusions) at the cost of more FPs (more alert fatigue). Given §6.5's asymmetric
  cost argument, this is usually the right direction to move for a production IDS — a missed
  attack is worse than an extra ticket in the SOC queue.
- **Raising the threshold** reduces FPs but increases FNs — appropriate only if the SOC is
  already overwhelmed and cannot act on additional alerts regardless of their validity.
- **ROC-AUC (§5.6) exists precisely to evaluate this trade-off independent of any one
  threshold** — a higher ROC-AUC model has a better FN/FP frontier available to tune into,
  even if its default-threshold F1/accuracy is not the highest.
- **Practical recommendation:** given §6.2's finding that some attack types are missed
  regardless of model, threshold tuning alone cannot fully close the recall gap on those
  types — it only trades errors *within* the attacks the feature set can already discriminate.
  Closing the gap on the collapsed attack types (§6.2, §6.3) would require either additional
  features that discriminate those specific flows, or a lower operating threshold accepted
  together with its FP cost.

### §6 Complete — Summary

| Step | Status | Key output |
|------|--------|------------|
| §6.1 Setup + attack-type recovery | ✅ | `y_test_multiclass.joblib` |
| §6.2 Per-attack-type recall | ✅ | `error_by_attack_type.png` |
| §6.3 Sample false negatives | ✅ | worst-detected attack type inspected |
| §6.4 Sample false positives | ✅ | FP counts per model + example flows |
| §6.5 Cybersecurity implications | ⚠️ | drafted — rewrite in your own words |
| §6.6 FP/FN trade-off | ⚠️ | drafted — rewrite in your own words |

**Next:** Phase 8 — your own experiments (overfitting-vs-concept-drift check, real-imbalanced
re-evaluation) build directly on the per-attack-type recall pattern established here.

---
## Phase 8 — Your Experiments

*(To be completed in Colab after §5 and §6 are done.)*

---
## §7 — Executive Summary

*(Filled after all analysis is complete.)*

---
## §8 — Summing It Up

*(Filled after all analysis is complete.)*